# Solar filament segmentation: cached inputs for Kaggle 2 x T4

This self-contained notebook targets **Kaggle GPU T4 x2 (16 GB per GPU)**. Select
that accelerator before Run All. It preserves full 2048 x 2048 images, the complete
preprocessing pipeline, the YOLO11s-seg architecture, the configured epoch budget,
date-grouped train/calibration/report splits, and the audited submission format.

**Before training:** parallel CPU workers prepare original/enhanced/candidate model
inputs once and store lossless uint8 BGR NumPy arrays. The training loop reads these
uncompressed arrays instead of rerunning Hough detection, background fitting, CLAHE,
and thresholding every epoch. Source images and annotations stay untouched.

**During training:** both T4s use DDP and AMP; a bounded VRAM probe on the T4 with
less free memory chooses one common per-GPU batch size, capped at four. Two loader
workers per rank, four prefetched batches per worker, pinned memory, persistent
iterators, and a separate CUDA copy stream prepare the next batch while the current
batch computes. Fixed-shape cuDNN autotuning favors throughput. Seeds remain fixed;
strict deterministic execution is disabled. No local-machine performance tuning is used.

File pages can be warmed into the shared OS page cache when RAM permits. Each rank
reads the same cache without independent full-dataset Python RAM copies. Calibration
reuses prepared inputs; report/test use the same deterministic preprocessing.

The notebook records GPU utilization and per-epoch input waiting time. These expose
remaining stalls; continuous 100% utilization is not guaranteed by a loader setting.
After successful submission/audits and previews, generated inputs, labels, and
prediction maps are deleted. Weights, training history, throughput/utilization logs,
metrics, configuration/runtime, audits, and `a_submission.csv` are retained.

## Quick repair of an existing CSV (no retraining or model inference)

If the previous submission was rejected for overlapping masks, run **only the next cell** after setting `RUN_REPAIR_EXISTING_CSV=True` and `REPAIR_INPUT_CSV` to that CSV. It imports its own libraries and does not need any trained model, dataset paths or other notebook cells. The source CSV is preserved; upload the new `*_nonoverlap.csv` file.

For each image, this applies the original union -> 8-connected components operation, re-encodes each component using the original COCO counts format, and numbers the rows again. The default repair threshold is 1 pixel to preserve the input foreground; set `min_area=20` in the call to reproduce the original notebook's component filter too. A disk-backed spool handles interleaved image rows without storing all masks in RAM. The repair audits every written RLE for overlap and writes a separate JSON report.

After a fresh run of the full notebook, use its normal submission; the final pipeline now enforces this same non-overlap rule. Old scores selected using the invalid instance mode should be recalibrated; no new score is claimed by repairing the CSV.


In [ ]:
# Standalone recovery: this cell does not train a model or run GPU inference.
import csv
import json
import os
import sqlite3
import tempfile
from contextlib import closing
from pathlib import Path
import cv2
import numpy as np
from pycocotools import mask as mask_utils


def repair_submission_overlaps(input_path, output_path=None, min_area=1):
    input_path = Path(input_path)
    output_path = (
        Path(output_path)
        if output_path is not None
        else input_path.with_name(input_path.stem + "_nonoverlap.csv")
    )
    if input_path.resolve() == output_path.resolve():
        raise ValueError(
            "Choose a separate output file; the rejected CSV is preserved."
        )
    if min_area < 1:
        raise ValueError("min_area must be at least 1 pixel.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    csv.field_size_limit(16 * 1024 * 1024)

    def encode_counts(binary_mask):
        # Exact encoding operation from the original 0.36 notebook.
        encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
        return encoded["counts"].decode("utf-8")

    report = dict(
        input_csv=str(input_path),
        output_csv=str(output_path),
        input_rows=0,
        output_rows=0,
        images_processed=0,
        images_with_input_overlaps=0,
        input_overlap_assignments=0,
        min_area=int(min_area),
        non_overlapping_masks=False,
    )
    with tempfile.TemporaryDirectory(
        prefix="filament_nonoverlap_", dir=output_path.parent
    ) as directory:
        temporary_csv = Path(directory) / "submission.csv"
        with closing(
            sqlite3.connect(str(Path(directory) / "input.sqlite"))
        ) as connection:
            connection.execute(
                "CREATE TABLE masks (filament_id TEXT PRIMARY KEY, stem TEXT, counts TEXT)"
            )
            with input_path.open(newline="", encoding="utf-8") as stream:
                reader = csv.DictReader(stream)
                if reader.fieldnames != ["filament_id", "segmentation_rle"]:
                    raise ValueError(
                        "Expected exactly filament_id,segmentation_rle columns."
                    )
                for row in reader:
                    if (
                        set(row) != {"filament_id", "segmentation_rle"}
                        or None in row.values()
                    ):
                        raise ValueError("Malformed submission row.")
                    fid, counts = row["filament_id"], row["segmentation_rle"]
                    if not fid or not counts or "_" not in fid:
                        raise ValueError("Empty or invalid filament ID / RLE counts.")
                    stem, suffix = fid.rsplit("_", 1)
                    if not stem or not suffix:
                        raise ValueError(f"Invalid filament ID: {fid}")
                    try:
                        connection.execute(
                            "INSERT INTO masks VALUES (?, ?, ?)", (fid, stem, counts)
                        )
                    except sqlite3.IntegrityError as error:
                        raise ValueError(f"Duplicate filament ID: {fid}") from error
                    report["input_rows"] += 1
            connection.execute("CREATE INDEX masks_stem ON masks(stem)")
            connection.commit()
            with temporary_csv.open("w", newline="", encoding="utf-8") as stream:
                writer = csv.writer(stream)  # Same default writer as original notebook.
                writer.writerow(["filament_id", "segmentation_rle"])
                for (stem,) in connection.execute(
                    "SELECT DISTINCT stem FROM masks ORDER BY stem"
                ):
                    union = np.zeros((2048, 2048), np.uint8)
                    overlap_assignments = 0
                    for (counts,) in connection.execute(
                        "SELECT counts FROM masks WHERE stem=?", (stem,)
                    ):
                        rle = {"size": [2048, 2048], "counts": counts.encode("utf-8")}
                        mask = mask_utils.decode(rle).astype(np.uint8)
                        if mask.shape != (2048, 2048) or not mask.any():
                            raise ValueError(f"Empty or invalid mask for {stem}")
                        overlap_assignments += int(np.count_nonzero(union & mask))
                        union |= mask
                    number, labels, stats, _ = cv2.connectedComponentsWithStats(
                        union, connectivity=8
                    )
                    keep = [
                        i
                        for i in range(1, number)
                        if stats[i, cv2.CC_STAT_AREA] >= min_area
                    ]
                    keep.sort(
                        key=lambda i: (
                            -int(stats[i, cv2.CC_STAT_AREA]),
                            int(stats[i, cv2.CC_STAT_TOP]),
                            int(stats[i, cv2.CC_STAT_LEFT]),
                        )
                    )
                    for counter, label in enumerate(keep, start=1):
                        writer.writerow(
                            [f"{stem}_{counter}", encode_counts(labels == label)]
                        )
                    report["images_processed"] += 1
                    report["output_rows"] += len(keep)
                    report["images_with_input_overlaps"] += int(overlap_assignments > 0)
                    report["input_overlap_assignments"] += overlap_assignments

        # Independently verify the actual written RLEs before publishing the CSV.
        occupied = {}
        seen = set()
        with temporary_csv.open(newline="", encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                fid = row["filament_id"]
                if fid in seen:
                    raise ValueError(f"Duplicate output ID: {fid}")
                seen.add(fid)
                stem = fid.rsplit("_", 1)[0]
                rle = {
                    "size": [2048, 2048],
                    "counts": row["segmentation_rle"].encode("utf-8"),
                }
                mask = mask_utils.decode(rle)
                if (
                    mask.shape != (2048, 2048)
                    or not mask.any()
                    or encode_counts(mask) != row["segmentation_rle"]
                ):
                    raise ValueError(f"Invalid repaired mask: {fid}")
                if stem in occupied:
                    overlap = mask_utils.merge([occupied[stem], rle], intersect=True)
                    if int(mask_utils.area(overlap)):
                        raise ValueError(f"Repaired masks still overlap: {fid}")
                    occupied[stem] = mask_utils.merge([occupied[stem], rle])
                else:
                    occupied[stem] = rle
        if len(seen) != report["output_rows"]:
            raise ValueError("Repaired CSV row-count mismatch.")
        os.replace(temporary_csv, output_path)
    report["non_overlapping_masks"] = True
    report["output_overlap_pixels"] = 0
    output_path.with_suffix(".audit.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    return output_path, report


RUN_REPAIR_EXISTING_CSV = False
REPAIR_INPUT_CSV = Path(
    "solar_filament_yolo11s_submission.csv"
)  # Or Path("submission.csv").
if RUN_REPAIR_EXISTING_CSV:
    repaired_csv_path, repair_report = repair_submission_overlaps(REPAIR_INPUT_CSV)
    print(json.dumps(repair_report, indent=2))
    print(f"Upload this corrected CSV: {repaired_csv_path}")

## What changes in this experiment

The former bilateral-only image transform is replaced by the preprocessing notebook's
complete deterministic pipeline. Its numeric stages and defaults are embedded
verbatim; exports and diagnostic float arrays are omitted during model loading.
The source hash, algorithm version, parameters, and channel mode enter the run
fingerprint, preventing reuse of earlier bilateral checkpoints or maps.

YOLO uses BGR arrays internally and converts them to RGB tensors. Both the training
loader and NumPy inference source therefore receive `[candidates, enhanced,
original]` in BGR storage, yielding the intended tensor channels. HSV changes,
channel swaps, and automatic color augmentations are disabled. Flips still transform
all feature channels and their instance labels together.

All annotators of one physical image remain in the same date group. Only supplied
training annotations create polygon targets. Preprocessing does not accept labels;
per-image normalization uses on-disk pixels, with no dataset-wide fitting. Candidate
thresholding uses the reference's documented upper-edge adaptation; `threshold_edge`
can reproduce the lower-edge rule. No public/test labels or pseudo-labels are used.

Calibration selects the checkpoint and postprocessing rule. The separate report
partition is evaluated after selection; neither local score is a public score. The
existing evaluator and optional probability-based postprocessing are retained.
Morphology guidance now uses the pipeline's candidate feature, restricted to pixels
already supported by YOLO probabilities. It cannot generate independent predictions.

## 1. Dependencies

Keep Ultralytics pinned to **8.4.171** and Kaggle's existing CUDA-enabled Torch.
Pillow provides the same source validation/grayscale loading as the preprocessing
notebook. The embedded trainer/validator target this pinned release. Resolved
versions and preprocessing source hashes are saved with every experiment.

In [ ]:
%pip -q install --no-cache-dir ultralytics==8.4.171 pycocotools scikit-learn scipy tqdm threadpoolctl psutil pillow cloudpickle

In [ ]:
from pathlib import Path
import csv, gc, hashlib, importlib, json, os, queue, random, re, shutil, subprocess, sys, time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version

for _thread_key in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[_thread_key] = "1"  # Also inherited by Ultralytics DDP subprocesses.
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import ultralytics
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from sklearn.model_selection import GroupShuffleSplit
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.engine.results import Results
from ultralytics.models.yolo.segment import SegmentationPredictor, SegmentationTrainer
from ultralytics.utils import ops

assert (
    ultralytics.__version__ == "8.4.171"
), "Restart the kernel after installing the pinned version."
csv.field_size_limit(16 * 1024 * 1024)
cv2.setNumThreads(1)
torch.set_num_threads(1)
CPU_THREAD_LIMIT = threadpool_limits(limits=1)
from IPython.display import display


## 2. Dual-T4 throughput settings

Select Kaggle **GPU T4 x2**. The default guard stops a full run on other hardware.
`T4_BATCH_PER_GPU=0` probes batch sizes up to 4 on the T4 with less free VRAM, using
70% as the profiling target. It then supplies a positive integer global batch to DDP.
Set a positive override only after inspecting actual T4 memory logs. The same batch
fits both ranks; inference uses independent replicas on the two devices.

`PRECOMPUTE_INPUTS=True` prepares native uint8 inputs before training. The temporary
cache is about 13 MB per physical image, including a packed auxiliary candidate map.
Disk preflight budgets it alongside weights, prediction maps, and a 6 GiB reserve.
`WARM_OS_PAGE_CACHE=True` warms shared pages only when RAM permits. Loader workers
read memory-mapped arrays, copy each sample for safe geometric augmentation, and
prefetch batches. `TRAIN_PREFETCH_FACTOR=4` and `CUDA_BATCH_PREFETCH=True` overlap
CPU loading and host-to-device transfers with GPU computation. No filters run in
the cached training/validation loaders. The pretrained feature channel order is
unchanged; HSV/channel-shuffle transforms remain disabled.

Four total training workers are divided over the two ranks. Validation uses a small
prefetching pool and batch up to two; sleeping training workers remain persistent.
The default stream policy deletes prediction maps after scoring, retaining bounded
preview maps until display. `CLEANUP_INTERMEDIATES=True` removes both prepared inputs
and prediction maps only after successful completion; failures retain them for recovery.

If AutoBatch produces an ill-conditioned fit or an out-of-range fallback, the
notebook records the reason in `batch_plan.json` and selects one image per T4
(global batch 2). Set `T4_BATCH_PER_GPU=1` to bypass automatic profiling explicitly.

In [ ]:
SEED = 42
DATA_ROOT_OVERRIDE = ""  # Optional explicit competition root.
OUTPUT_ROOT = Path.cwd()
PREPROCESSING_METHOD = "solar_pipeline"
MODEL_INPUT_MODE = "multi_channel"  # multi_channel, enhanced, original
SOURCE_PIPELINE_VERSION = 'solar-filament-preprocessing-v3'
PREPROCESSING_CONFIG = {'expected_shape': [2048, 2048],
 'strict_shape': True,
 'hough_coarse_size': 512,
 'hough_fine_size': 1024,
 'hough_param1': 80.0,
 'hough_param2': 35.0,
 'disk_radius_fraction': [0.25, 0.49],
 'median_kernel': 3,
 'radial_bins': 128,
 'radial_sample_stride': 4,
 'radial_gain_limits': [0.5, 2.5],
 'polynomial_degree': 3,
 'polynomial_sample_stride': 16,
 'polynomial_clip_sigma': 3.0,
 'polynomial_iterations': 3,
 'clahe_clip_limit': 2.0,
 'clahe_grid': [8, 8],
 'normalization_percentiles': [1.0, 99.5],
 'threshold_intervals': 20,
 'threshold_sigma_span': 3.0,
 'threshold_edge': 'upper',
 'candidate_limb_margin_px': 2,
 'min_component_area_px': 16,
 'min_component_disk_fraction': 1e-05,
 'closing_kernel': 0,
 'save_intermediates': False,
 'save_float_stages': False}
CLEANUP_INTERMEDIATES = True
REQUIRE_TWO_T4 = True  # Kaggle accelerator: GPU T4 x2
PRECOMPUTE_INPUTS = True
WARM_OS_PAGE_CACHE = True
TRAIN_PREFETCH_FACTOR = 4
CUDA_BATCH_PREFETCH = True
VALIDATION_WORKERS = 2 if (os.cpu_count() or 1) >= 2 else 0
T4_BATCH_PER_GPU = 0  # 0 = probe the smaller/free-memory T4; otherwise an explicit positive integer
AUTO_BATCH_MEMORY_FRACTION = 0.70
AUTO_BATCH_MAX_PER_GPU = 4
SAVE_PREVIEW_IMAGES = False
SPLIT_GROUP = "date"
VALIDATION_FRACTION = 0.20
CALIBRATION_SHARE = 0.50
IMAGE_SIZE = 2048
YOLO_MODEL_NAME = "yolo11s-seg.pt"  # Previous model: "yolo26s-seg.pt".
MODEL_TAG = Path(YOLO_MODEL_NAME).stem.removesuffix("-seg")
NUM_EPOCHS = 100
PATIENCE = 5
TARGET_MIN_AREA = 20  # Same union-component threshold as the supplied notebook.
NUM_WORKERS = min(4, os.cpu_count() or 1)
GPU_IDS = list(range(min(2, torch.cuda.device_count())))
TRAIN_DEVICE = GPU_IDS if len(GPU_IDS) > 1 else (GPU_IDS[0] if GPU_IDS else "cpu")
INFERENCE_DEVICES = [f"cuda:{i}" for i in GPU_IDS] or ["cpu"]
BATCH_SIZE = max(1, len(GPU_IDS))
YOLO_WORKERS = NUM_WORKERS // max(1, len(GPU_IDS)) if GPU_IDS else 0
assert YOLO_WORKERS * max(1, len(GPU_IDS)) <= 4
MAX_DETECTIONS = 300 # Previous: 100
RAM_RESERVE_GIB = 6.0
DISK_RESERVE_GIB = 6.0
DISK_LIMIT_GIB = 57.6
PREDICTION_CACHE_POLICY = (
    "stream"  # stream per image; all caches are temporary by default.
)
RUN_TRAINING = True
EXISTING_CHECKPOINT = (
    ""  # Only used if RUN_TRAINING=False; matching run_config.json required.
)
CREATE_SUBMISSION = True
REBUILD_YOLO_DATASET = (
    False  # Fingerprints prevent reusing another experiment's dataset.
)
PREDICTION_PREVIEW_COUNT = 10
PREPROCESS_PREVIEW_COUNT = 3
CONFIDENCE_CANDIDATES = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
AREA_CANDIDATES = [0, 20, 80, 200]
TTA_VIEWS = 8  # 1, 2, 4 or 8; exact 90-degree rotations/reflections, no interpolation.
MASK_DECODE_CHUNK = 4  # At most four full-size float instance masks per GPU.
KEEP_RULES = [
    dict(t_lo=0.35, t_hi=0.70, min_meanp=0.50, min_area=80),
    dict(t_lo=0.35, t_hi=0.85, min_meanp=0.65, min_area=200),
    dict(t_lo=0.50, t_hi=0.90, min_meanp=0.75, min_area=200),
    dict(t_lo=0.35, t_hi=0.99, min_meanp=0.85, min_area=200),  # U-Net reference rule.
]
MORPHOLOGY_RULES = [
    dict(t_lo=0.35, t_hi=0.75, min_meanp=0.55, min_area=165, bridge_px=3),
    dict(t_lo=0.35, t_hi=0.85, min_meanp=0.65, min_area=200, bridge_px=5),
]
SAVE_EVERY_EPOCH = False
CLIP_TO_DISK_CANDIDATES = [False, True]
PREDICT_NMS_IOU = (
    0.70  # Fixed; not described as an optimal or instance-merging threshold.
)
PQ_MATCH_IOU = 0.50
GIB = 1024**3

TRAIN_OPTIONS = dict(
    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=TRAIN_DEVICE,
    workers=YOLO_WORKERS,
    patience=PATIENCE,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=0.5,
    nbs=16,
    save_period=1 if SAVE_EVERY_EPOCH else -1,
    amp=bool(GPU_IDS),
    seed=SEED,
    deterministic=False,  # fixed-shape cuDNN autotuning favors throughput
    max_det=MAX_DETECTIONS,
    fliplr=0.5,
    flipud=0.5,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    bgr=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=0,
    overlap_mask=True,
    mask_ratio=4,
    augmentations=[],
    plots=False,  # render history after training, not batch previews inside the GPU loop
    verbose=True,
)
assert PRECOMPUTE_INPUTS, 'This configuration prepares inputs before training.'
assert TRAIN_PREFETCH_FACTOR >= 1
assert T4_BATCH_PER_GPU >= 0
assert 0 < AUTO_BATCH_MEMORY_FRACTION < 1
assert PREDICTION_CACHE_POLICY in {"stream", "bounded", "all"}
assert TTA_VIEWS in {1, 2, 4, 8}
assert PREPROCESSING_METHOD == "solar_pipeline"
assert MODEL_INPUT_MODE in {"multi_channel", "enhanced", "original"}
assert SPLIT_GROUP in {"date", "filename"}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if GPU_IDS:
    torch.cuda.manual_seed_all(SEED)
print("Pretrained model:", YOLO_MODEL_NAME)
print(
    "Train devices:",
    TRAIN_DEVICE,
    "| batch:",
    BATCH_SIZE,
    "| inference:",
    INFERENCE_DEVICES,
)
if not GPU_IDS and RUN_TRAINING:
    print("CPU fallback detected. Full-resolution training will be slow.")

print(
    f"CPU workers: {NUM_WORKERS}; DDP loaders: {YOLO_WORKERS} per rank; validation loader workers: {VALIDATION_WORKERS}; TTA: {TTA_VIEWS}"
)

# Embedded implementation; configuration is above. Kept here so Kaggle needs only this notebook.

# Shared preprocessing, prepared-input loader, and CUDA prefetch implementation.
PREPROCESSING_RUNTIME_SOURCE = r'''"""Native solar preprocessing plus uncached YOLO loaders; embedded by the notebook."""
import copy
import csv
import torch
import json
import math
import time
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import get_split_fraction
from ultralytics.models.yolo.segment import SegmentationTrainer, SegmentationValidator
from ultralytics.utils import colorstr
from ultralytics.utils.torch_utils import unwrap_model

cv2.setNumThreads(1)
LAPLACIAN_KERNEL = np.array([[1, 1, 1], [1, -8, 1], [1, 1, 1]], np.float32)


def load_grayscale(path, cfg, coco_shape=None):
    # Check the source dtype rather than silently truncating a 16-bit image to uint8.
    with Image.open(path) as image:
        image.verify()
    with Image.open(path) as image:
        array = np.asarray(image)
        if array.dtype != np.uint8:
            raise ValueError(f"{path}: expected uint8 source, got {array.dtype}")
        gray = np.asarray(image.convert("L")).copy()
    if gray.ndim != 2 or not gray.size or not np.isfinite(gray).all():
        raise ValueError(f"{path}: expected a nonempty, finite 2-D image")
    if cfg["strict_shape"] and tuple(gray.shape) != tuple(cfg["expected_shape"]):
        raise ValueError(f'{path}: expected {cfg["expected_shape"]}, got {gray.shape}')
    if coco_shape and tuple(gray.shape) != tuple(coco_shape):
        raise ValueError(f"{path}: source shape disagrees with COCO metadata")
    return np.ascontiguousarray(gray)


def resize_for_detection(image, long_side):
    scale = min(1.0, long_side / max(image.shape))
    size = (
        max(1, round(image.shape[1] * scale)),
        max(1, round(image.shape[0] * scale)),
    )
    small = cv2.resize(image, size, interpolation=cv2.INTER_AREA)
    # Separate scale factors account for integer rounding on non-square inputs.
    return small, np.array([size[0] / image.shape[1], size[1] / image.shape[0]])


def valid_circle(cx, cy, radius, shape, cfg):
    h, w = shape
    lo, hi = cfg["disk_radius_fraction"]
    return (
        lo * min(h, w) <= radius <= hi * min(h, w)
        and abs(cx - (w - 1) / 2) < 0.20 * w
        and abs(cy - (h - 1) / 2) < 0.20 * h
        and cx - radius >= 0
        and cx + radius < w
        and cy - radius >= 0
        and cy + radius < h
    )


def detect_solar_disk(image, cfg):
    if float(np.std(image)) < 1:
        raise ValueError("Solar disk detection failed: image has almost no contrast.")
    coarse, scale = resize_for_detection(image, cfg["hough_coarse_size"])
    blur = cv2.medianBlur(coarse, 5)
    lo, hi = cfg["disk_radius_fraction"]
    circles = cv2.HoughCircles(
        blur,
        cv2.HOUGH_GRADIENT,
        dp=1.2,
        minDist=min(blur.shape) / 2,
        param1=cfg["hough_param1"],
        param2=cfg["hough_param2"],
        minRadius=round(lo * min(blur.shape)),
        maxRadius=round(hi * min(blur.shape)),
    )
    disk = None
    if circles is not None:
        for cx, cy, radius in circles[0]:
            values = (
                float(cx / scale[0]),
                float(cy / scale[1]),
                float(radius / scale.mean()),
            )
            if valid_circle(*values, image.shape, cfg):
                disk = dict(zip(("cx", "cy", "radius"), values), method="hough-coarse")
                break
    if disk is None:
        _, binary = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if contours:
            contour = max(contours, key=cv2.contourArea)
            if len(contour) >= 20 and cv2.contourArea(contour) > 0.15 * blur.size:
                (cx, cy), (a, b), _ = cv2.fitEllipse(contour)
                values = (
                    float(cx / scale[0]),
                    float(cy / scale[1]),
                    float((a + b) / (4 * scale.mean())),
                )
                if min(a, b) / max(a, b) > 0.90 and valid_circle(
                    *values, image.shape, cfg
                ):
                    disk = dict(
                        zip(("cx", "cy", "radius"), values), method="contour-fallback"
                    )
    if disk is None:
        raise ValueError(
            "No plausible solar disk found; inspect image/detection settings."
        )

    if disk["method"].startswith("hough"):
        fine, fs = resize_for_detection(image, cfg["hough_fine_size"])
        r = disk["radius"] * fs.mean()
        tolerance = max(4, 0.025 * r)
        refined = cv2.HoughCircles(
            cv2.medianBlur(fine, 5),
            cv2.HOUGH_GRADIENT,
            dp=1.2,
            minDist=min(fine.shape) / 2,
            param1=cfg["hough_param1"],
            param2=cfg["hough_param2"],
            minRadius=max(1, round(r - tolerance)),
            maxRadius=round(r + tolerance),
        )
        if refined is not None:
            for cx, cy, radius in refined[0]:
                values = (
                    float(cx / fs[0]),
                    float(cy / fs[1]),
                    float(radius / fs.mean()),
                )
                displacement = math.hypot(
                    values[0] - disk["cx"], values[1] - disk["cy"]
                )
                if displacement < 0.04 * disk["radius"] and valid_circle(
                    *values, image.shape, cfg
                ):
                    disk = dict(
                        zip(("cx", "cy", "radius"), values),
                        method="hough-coarse-to-fine",
                    )
                    break
    yy, xx = np.ogrid[: image.shape[0], : image.shape[1]]
    rho = (np.hypot(xx - disk["cx"], yy - disk["cy"]) / disk["radius"]).astype(
        np.float32
    )
    mask = rho <= 1.0
    if np.median(image[mask]) <= np.median(image[~mask]) + 5:
        raise ValueError(
            "Detected disk is not brighter than space; inspect this image."
        )
    return disk, rho, mask


def robust_unit(image, mask, percentiles):
    values = np.asarray(image)[mask]
    if not values.size or not np.isfinite(values).all():
        raise ValueError("Normalization requires finite on-disk pixels.")
    low, high = np.percentile(values, percentiles)
    result = np.zeros(image.shape, np.float32)
    if high - low <= 1e-6:
        result[mask] = 0.5
    else:
        result[mask] = np.clip((image[mask] - low) / (high - low), 0, 1)
    return result, {"low": float(low), "high": float(high)}


def to_uint8(unit):
    return np.rint(np.clip(unit, 0, 1) * 255).astype(np.uint8)


def radial_correction(image, rho, disk_mask, cfg):
    stride = cfg["radial_sample_stride"]
    radii = rho[::stride, ::stride]
    intensities = image[::stride, ::stride].astype(np.float32)
    bins = cfg["radial_bins"]
    centers = (np.arange(bins) + 0.5) / bins
    profile = np.full(bins, np.nan, np.float32)
    for i in range(bins):
        pixels = intensities[(radii >= i / bins) & (radii < (i + 1) / bins)]
        if pixels.size >= 4:
            profile[i] = np.median(pixels)
    good = np.isfinite(profile) & (profile > 1)
    if good.sum() < max(4, bins // 4):
        raise ValueError(
            "Insufficient radial samples; adjust radial sampling settings."
        )
    profile = np.interp(centers, centers[good], profile[good]).astype(np.float32)
    profile = cv2.GaussianBlur(profile[None, :], (9, 1), 0).ravel()
    target = float(np.median(profile[centers < 0.5]))
    background = (
        np.interp(rho.ravel(), centers, profile).reshape(image.shape).astype(np.float32)
    )
    gain = np.clip(target / np.maximum(background, 1), *cfg["radial_gain_limits"])
    corrected = image.astype(np.float32) * gain
    corrected[~disk_mask] = 0
    return corrected, centers, profile


def polynomial_correction(image, disk_mask, disk, cfg):
    yy, xx = np.ogrid[: image.shape[0], : image.shape[1]]
    x = ((xx - disk["cx"]) / disk["radius"]).astype(np.float32)
    y = ((yy - disk["cy"]) / disk["radius"]).astype(np.float32)
    stride = cfg["polynomial_sample_stride"]
    sampled_mask = disk_mask[::stride, ::stride]
    xs = np.broadcast_to(x[:, ::stride], sampled_mask.shape)[sampled_mask]
    ys = np.broadcast_to(y[::stride, :], sampled_mask.shape)[sampled_mask]
    values = image[::stride, ::stride][sampled_mask].astype(np.float64)
    degree = cfg["polynomial_degree"]
    powers = [
        (i, j)
        for total in range(degree + 1)
        for i in range(total + 1)
        for j in [total - i]
    ]
    design = np.column_stack([xs**i * ys**j for i, j in powers]).astype(np.float64)
    low, high = np.percentile(values, [10, 90])
    keep = (values >= low) & (values <= high)
    coefficients = None
    for _ in range(cfg["polynomial_iterations"]):
        if keep.sum() < 3 * len(powers):
            raise ValueError("Insufficient pixels for polynomial background fit.")
        coefficients, _, rank, _ = np.linalg.lstsq(
            design[keep], values[keep], rcond=None
        )
        if rank != len(powers):
            raise ValueError("Polynomial background fit is rank deficient.")
        residual = values - design @ coefficients
        center = np.median(residual[keep])
        mad = max(1.4826 * np.median(np.abs(residual[keep] - center)), 1e-3)
        keep = np.abs(residual - center) <= cfg["polynomial_clip_sigma"] * mad
    background = np.zeros(image.shape, np.float32)
    for coefficient, (i, j) in zip(coefficients, powers):
        background += np.float32(coefficient) * (x**i) * (y**j)
    target = float(np.median(background[disk_mask]))
    corrected = image - background + target
    corrected[~disk_mask] = 0
    background[~disk_mask] = 0
    return corrected.astype(np.float32), background, coefficients.tolist()


def enhance_image(corrected, disk_mask, cfg):
    unit, before_stats = robust_unit(
        corrected, disk_mask, cfg["normalization_percentiles"]
    )
    clahe_input = to_uint8(unit)
    # Neutral off-disk fill limits black-space influence in edge CLAHE tiles.
    clahe_input[~disk_mask] = int(np.median(clahe_input[disk_mask]))
    clahe = cv2.createCLAHE(
        clipLimit=cfg["clahe_clip_limit"], tileGridSize=tuple(cfg["clahe_grid"])
    ).apply(clahe_input)
    clahe[~disk_mask] = 0
    normalized, after_stats = robust_unit(
        clahe, disk_mask, cfg["normalization_percentiles"]
    )
    return (
        clahe,
        to_uint8(normalized),
        {"pre_clahe": before_stats, "post_clahe": after_stats},
    )


def adaptive_candidates(enhanced, disk_mask, cfg):
    unit = enhanced.astype(np.float32) / 255.0
    laplacian = cv2.filter2D(
        unit, cv2.CV_32F, LAPLACIAN_KERNEL, borderType=cv2.BORDER_REFLECT_101
    )
    margin = int(cfg["candidate_limb_margin_px"])
    if margin < 0:
        raise ValueError("Candidate limb margin cannot be negative.")
    eligible = (
        cv2.erode(
            disk_mask.astype(np.uint8), np.ones((3, 3), np.uint8), iterations=margin
        ).astype(bool)
        if margin
        else disk_mask.copy()
    )
    laplacian[~eligible] = 0
    values = unit[eligible]
    if not values.size:
        raise ValueError("Candidate mask has no interior pixels.")
    median, std = float(np.median(values)), float(np.std(values))
    intervals = int(cfg["threshold_intervals"])
    if intervals < 2:
        raise ValueError("Use at least two threshold intervals.")
    if cfg["threshold_edge"] not in ("lower", "upper"):
        raise ValueError("threshold_edge must be lower or upper.")
    edges = np.linspace(
        median - cfg["threshold_sigma_span"] * std, median, intervals + 1
    )
    scores = np.full(intervals, np.nan)
    counts = np.zeros(intervals, np.int64)
    responses = laplacian[eligible]
    for i, (low, high) in enumerate(zip(edges[:-1], edges[1:])):
        included = (values >= low) & (values < high)
        counts[i] = included.sum()
        if counts[i]:
            scores[i] = float(np.mean(responses[included]))
    selected = None
    raw = np.zeros(enhanced.shape, bool)
    if std > 1e-6 and np.isfinite(scores).any():
        best = int(np.nanargmax(scores))
        if scores[best] > 0:
            selected = float(edges[best + (cfg["threshold_edge"] == "upper")])
            raw = eligible & (unit < selected)
    diagnostics = {
        "median": median,
        "std": std,
        "threshold": selected,
        "threshold_edge": cfg["threshold_edge"],
        "edges": edges.tolist(),
        "counts": counts.tolist(),
        "mean_laplacian": [float(s) if np.isfinite(s) else None for s in scores],
    }
    return laplacian, eligible, raw, diagnostics


def filter_components(mask, disk_area, cfg):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    minimum = max(
        int(cfg["min_component_area_px"]),
        math.ceil(disk_area * cfg["min_component_disk_fraction"]),
    )
    keep = stats[:, cv2.CC_STAT_AREA] >= minimum
    keep[0] = False
    result = keep[labels]
    return result, {
        "minimum_area_px": minimum,
        "raw_components": count - 1,
        "filtered_components": int(keep.sum()),
    }


def conservative_closing(mask, eligible, cfg):
    size = int(cfg["closing_kernel"])
    if size == 0:
        return mask.copy()
    if size != 3:
        raise ValueError("Conservative closing supports 0 (off) or 3 (one iteration).")
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (size, size))
    return (
        cv2.morphologyEx(
            mask.astype(np.uint8), cv2.MORPH_CLOSE, kernel, iterations=1
        ).astype(bool)
        & eligible
    )


def preprocess_image(image, cfg):
    if image.dtype != np.uint8 or image.ndim != 2:
        raise ValueError("preprocess_image expects a 2-D uint8 grayscale array.")
    started = time.perf_counter()
    disk, rho, disk_mask = detect_solar_disk(image, cfg)
    masked = np.where(disk_mask, image, 0).astype(np.uint8)
    median = cv2.medianBlur(image, cfg["median_kernel"])
    median[~disk_mask] = 0
    radial, radial_centers, radial_profile = radial_correction(
        median, rho, disk_mask, cfg
    )
    corrected, background, coefficients = polynomial_correction(
        radial, disk_mask, disk, cfg
    )
    clahe, enhanced, normalization = enhance_image(corrected, disk_mask, cfg)
    laplacian, eligible, raw, thresholds = adaptive_candidates(enhanced, disk_mask, cfg)
    filtered, components = filter_components(raw, int(disk_mask.sum()), cfg)
    candidates = conservative_closing(filtered, eligible, cfg)
    multi_channel = np.stack(
        [image, enhanced, candidates.astype(np.uint8) * 255], axis=-1
    )
    return {
        "original": image.copy(),
        "disk_mask": disk_mask,
        "masked": masked,
        "median": median,
        "radial": radial,
        "background": background,
        "corrected": corrected,
        "clahe": clahe,
        "enhanced": enhanced,
        "laplacian": laplacian,
        "raw_candidates": raw,
        "filtered_candidates": filtered,
        "candidates": candidates,
        "multi_channel": multi_channel,
        "metadata": {
            "disk": disk,
            "shape": list(image.shape),
            "radial_centers": radial_centers.tolist(),
            "radial_profile": radial_profile.tolist(),
            "polynomial_coefficients": coefficients,
            "normalization": normalization,
            "threshold_selection": thresholds,
            "components": components,
            "preprocessing_seconds": time.perf_counter() - started,
        },
    }


def preprocess_model_input(image, config, mode='multi_channel', return_aux=False):
    """Return native uint8 BGR; YOLO's normal BGR->RGB yields [original, enhanced, candidates]."""
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError('Expected a native 2-D uint8 grayscale source.')
    if config['strict_shape'] and list(image.shape) != list(config['expected_shape']):
        raise ValueError(f'Unexpected source shape: {image.shape}')
    if mode not in ('multi_channel', 'enhanced', 'original'):
        raise ValueError('Input mode must be multi_channel, enhanced, or original.')
    if mode == 'original' and not return_aux:
        return cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    # Reference stages live only for this sample. Nothing is cached in a dataset or module.
    stages = preprocess_image(image, config)
    if mode == 'multi_channel':
        bgr = cv2.cvtColor(stages['multi_channel'], cv2.COLOR_RGB2BGR)
    else:
        gray = stages['enhanced'] if mode == 'enhanced' else image
        bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    if return_aux:
        aux = {'disk': stages['metadata']['disk'],
               'candidates': stages['candidates'].astype(np.uint8) * 255,
               'preprocessing_seconds': stages['metadata']['preprocessing_seconds']}
        return np.ascontiguousarray(bgr), aux
    return np.ascontiguousarray(bgr)


def read_cached_bgr(path, expected_shape):
    """Read shared OS-cached pages and return a private writable augmentation array."""
    if not Path(path).is_file():
        raise FileNotFoundError(f'Prepared input missing: {path}. Rebuild the input cache before training.')
    mapped = np.load(path, mmap_mode='r', allow_pickle=False)
    try:
        if mapped.dtype != np.uint8 or mapped.shape != (*expected_shape, 3):
            raise ValueError(f'Invalid prepared BGR input: {path}, {mapped.shape}, {mapped.dtype}')
        return np.array(mapped, copy=True, order='C')
    finally:
        if getattr(mapped, '_mmap', None) is not None:
            mapped._mmap.close()


class StreamingSolarDataset(YOLODataset):
    """Load prepared native inputs; preprocessing is never rerun in the cached training path."""

    def __init__(self, *args, preprocessing_config, input_mode, prepared_inputs=False,
                 input_cache_root=None, **kwargs):
        self.preprocessing_config = copy.deepcopy(preprocessing_config)
        self.input_mode = input_mode
        self.prepared_inputs = bool(prepared_inputs)
        self.input_cache_root = Path(input_cache_root) if input_cache_root else None
        kwargs['cache'] = False  # disable Ultralytics' separate raw/RAM image cache
        super().__init__(*args, **kwargs)

    def load_image(self, i, rect_mode=True, resize_short=False):
        if self.prepared_inputs:
            alias = Path(self.im_files[i])
            path = self.input_cache_root / alias.parent.name / (alias.stem + '.npy')
            im = read_cached_bgr(path, self.preprocessing_config['expected_shape'])
            h0, w0 = im.shape[:2]
        else:
            # Optional compatibility path for small uncached experiments.
            image = load_grayscale(self.im_files[i], self.preprocessing_config)
            im = preprocess_model_input(image, self.preprocessing_config, self.input_mode)
            h0, w0 = image.shape
        shape = tuple(self.imgsz) if isinstance(self.imgsz, (tuple, list)) else (self.imgsz, self.imgsz)
        size = max(shape)
        if rect_mode:
            ratio = size / (min(h0, w0) if resize_short else max(h0, w0))
            if ratio != 1:
                if resize_short:
                    w, h = ((math.ceil(w0 * ratio), size) if h0 < w0
                            else (size, math.ceil(h0 * ratio)))
                else:
                    w, h = min(math.ceil(w0 * ratio), size), min(math.ceil(h0 * ratio), size)
                im = cv2.resize(im, (w, h), interpolation=cv2.INTER_LINEAR)
        elif (h0, w0) != shape:
            im = cv2.resize(im, shape[::-1], interpolation=cv2.INTER_LINEAR)
        return np.ascontiguousarray(im), (h0, w0), im.shape[:2]


def loader_settings(args):
    path = Path(args.data).resolve().parent / 'streaming_config.json'
    return json.loads(path.read_text(encoding='utf-8'))


def build_streaming_dataset(args, image_path, batch, data, mode, stride):
    config_path = Path(args.data).resolve().parent / 'streaming_config.json'
    settings = loader_settings(args)
    return StreamingSolarDataset(
        img_path=image_path, imgsz=args.imgsz, batch_size=batch,
        augment=mode == 'train', hyp=args, rect=args.rect or mode == 'val',
        cache=False, single_cls=args.single_cls or False, stride=int(stride),
        pad=0.0 if mode == 'train' else 0.5, prefix=colorstr(f'{mode}: '),
        task=args.task, classes=args.classes, data=data,
        fraction=get_split_fraction(args.fraction, 'train' if mode == 'train' else args.split or 'val'),
        preprocessing_config=settings['preprocessing'], input_mode=settings['input_mode'],
        prepared_inputs=settings.get('prepared_inputs', False),
        input_cache_root=config_path.parent / 'input_cache',
    )


def move_batch(batch, device):
    # Preserve uint8 inputs. Ultralytics performs /255 once in its normal preprocess_batch.
    return {key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value
            for key, value in batch.items()}


class PrefetchLoader:
    """Reuse Ultralytics' persistent iterator, with one GPU batch prefetched on a CUDA stream."""
    def __init__(self, loader, device, cuda_prefetch=True):
        self.loader = loader
        self.device = torch.device(device)
        self.enabled = bool(cuda_prefetch and self.device.type == 'cuda')
        self.stats = {}

    def __getattr__(self, name):
        return getattr(self.loader, name)

    def __len__(self):
        return len(self.loader)

    def __iter__(self):
        iterator = iter(self.loader)
        self.stats = {'batches': 0, 'images': 0, 'input_wait_seconds': 0.0,
                      'started': time.perf_counter(), 'cuda_prefetch': self.enabled}
        def fetch():
            started = time.perf_counter()
            try:
                return next(iterator)
            except StopIteration:
                return None
            finally:
                self.stats['input_wait_seconds'] += time.perf_counter() - started
        def counted(batch):
            self.stats['batches'] += 1
            self.stats['images'] += len(batch['img'])
            return batch
        if not self.enabled:
            for _ in range(len(self.loader)):
                batch = fetch()
                if batch is None:
                    return
                yield counted(batch)
            return
        stream = torch.cuda.Stream(device=self.device)
        def preload():
            batch = fetch()
            if batch is not None:
                with torch.cuda.stream(stream):
                    batch = move_batch(batch, self.device)
            return batch
        next_batch = preload()
        while next_batch is not None:
            current = torch.cuda.current_stream(self.device)
            current.wait_stream(stream)
            batch = next_batch
            for value in batch.values():
                if isinstance(value, torch.Tensor):
                    value.record_stream(current)
            # Enqueue the next H2D copy after the wait dependency above; it can overlap compute.
            next_batch = preload()
            yield counted(batch)


def configure_prefetch(loader, device, settings, mode):
    if loader.num_workers:
        loader.prefetch_factor = int(settings.get('prefetch_factor', 4)) if mode == 'train' else 2
    return PrefetchLoader(loader, device, settings.get('cuda_prefetch', True))


def log_training_throughput(trainer):
    if torch.distributed.is_initialized() and torch.distributed.get_rank() != 0:
        return
    loader = trainer.train_loader
    stats = getattr(loader, 'stats', {})
    if not stats.get('batches'):
        return
    if trainer.device.type == 'cuda':
        torch.cuda.synchronize(trainer.device)
    elapsed = max(time.perf_counter() - stats['started'], 1e-6)
    row = dict(epoch=trainer.epoch + 1, rank_images=stats['images'],
               train_seconds=elapsed, rank_images_per_second=stats['images'] / elapsed,
               input_wait_ms_per_batch=1000 * stats['input_wait_seconds'] / stats['batches'],
               input_wait_fraction=stats['input_wait_seconds'] / elapsed,
               cuda_prefetch=stats['cuda_prefetch'])
    path = Path(trainer.save_dir) / 'input_throughput.csv'
    header = not path.exists()
    with path.open('a', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(row))
        if header:
            writer.writeheader()
        writer.writerow(row)


class StreamingSolarSegmentationValidator(SegmentationValidator):
    def build_dataset(self, img_path, mode='val', batch=None):
        return build_streaming_dataset(self.args, img_path, batch, self.data, mode, self.stride)

    def get_dataloader(self, dataset_path, batch_size):
        settings = loader_settings(self.args)
        previous = self.args.workers
        try:
            self.args.workers = int(settings.get('validation_workers', 0))
            loader = super().get_dataloader(dataset_path, int(settings.get('validation_batch_size', 1)))
            return configure_prefetch(loader, self.device, settings, 'val')
        finally:
            self.args.workers = previous


class StreamingSolarSegmentationTrainer(SegmentationTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if self.device.type == 'cuda' and not self.args.deterministic and not self.args.multi_scale:
            torch.backends.cudnn.benchmark = True  # fixed 2048-square tensors on T4
        self.add_callback('on_train_epoch_end', log_training_throughput)

    def build_dataset(self, img_path, mode='train', batch=None):
        stride = max(int(unwrap_model(self.model).stride.max()), 32)
        return build_streaming_dataset(self.args, img_path, batch, self.data, mode, stride)

    def get_validator(self):
        return StreamingSolarSegmentationValidator(
            self.test_loader, save_dir=self.save_dir,
            args=copy.copy(self.args), _callbacks=self.callbacks)

    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode='train'):
        settings = loader_settings(self.args)
        previous = self.args.workers
        try:
            if mode != 'train':
                # The pinned training loader doubles args.workers for validation.
                self.args.workers = int(settings.get('validation_workers', 0)) // 2
                batch_size = int(settings.get('validation_batch_size', 1))
            loader = super().get_dataloader(dataset_path, batch_size, rank, mode)
            return configure_prefetch(loader, self.device, settings, mode)
        finally:
            self.args.workers = previous
'''
PREPROCESSING_RUNTIME_SHA256 = hashlib.sha256(PREPROCESSING_RUNTIME_SOURCE.encode()).hexdigest()

In [ ]:
def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=True), encoding="utf-8")
    os.replace(tmp, path)


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    roots = (
        [Path(DATA_ROOT_OVERRIDE)]
        if DATA_ROOT_OVERRIDE
        else [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]
    )
    for parent in roots:
        if not parent.is_dir():
            continue
        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent
            if (root / "train/train_images").is_dir() and (
                root / "test/test_images"
            ).is_dir():
                return root.resolve()
    raise FileNotFoundError("Attach the competition dataset or set DATA_ROOT_OVERRIDE.")


def read_gray(path):
    # The imported runtime is initialized before any image is loaded.
    return solar_runtime.load_grayscale(path, PREPROCESSING_CONFIG)

def bounded_map(function, items, workers=NUM_WORKERS):
    iterator = iter(items)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending = deque()
        for _ in range(workers):
            item = next(iterator, None)
            if item is not None:
                pending.append(pool.submit(function, item))
        while pending:
            yield pending.popleft().result()  # Propagate every worker failure.
            item = next(iterator, None)
            if item is not None:
                pending.append(pool.submit(function, item))


DATA_ROOT = find_dataset_root()
TRAIN_IMAGE_DIR = DATA_ROOT / "train/train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test/test_images"
ANNOTATION_PATH = DATA_ROOT / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
coco = COCO(str(ANNOTATION_PATH))
records = pd.DataFrame(
    [
        dict(
            image_id=i,
            file_name=m["file_name"],
            date=str(m.get("date_captured", ""))[:10],
        )
        for i, m in coco.imgs.items()
    ]
)
if SPLIT_GROUP == "date":
    records["group"] = records["file_name"].map(
        lambda n: re.match(r"^(\d{8})", Path(n).name).group(1)
    )
else:
    records["group"] = records["file_name"]
splitter = GroupShuffleSplit(
    n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED
)
train_idx, holdout_idx = next(splitter.split(records, groups=records["group"]))
train_records = records.iloc[train_idx].reset_index(drop=True)
holdout = records.iloc[holdout_idx].reset_index(drop=True)
splitter2 = GroupShuffleSplit(
    n_splits=1, train_size=CALIBRATION_SHARE, random_state=SEED + 1
)
cal_idx, report_idx = next(splitter2.split(holdout, groups=holdout["group"]))
calibration_records = holdout.iloc[cal_idx].reset_index(drop=True)
report_records = holdout.iloc[report_idx].reset_index(drop=True)
for left, right in [
    (train_records, calibration_records),
    (train_records, report_records),
    (calibration_records, report_records),
]:
    assert set(left.file_name).isdisjoint(right.file_name)
    assert set(left.group).isdisjoint(right.group)


def physical_rows(frame):
    return (
        frame.groupby("file_name", sort=True)["image_id"]
        .agg(list)
        .reset_index(name="image_ids")
        .to_dict("records")
    )


def choose_dual_t4_batch():
    if REQUIRE_TWO_T4:
        names = [torch.cuda.get_device_name(i) for i in GPU_IDS]
        if len(GPU_IDS) != 2 or not all('T4' in name.upper() for name in names):
            raise RuntimeError('Select Kaggle accelerator GPU T4 x2 before Run All. No local-machine tuning is used.')
    devices = len(GPU_IDS)
    if not devices:
        return 1, {'per_gpu': 1, 'reason': 'CPU fallback explicitly enabled'}
    if not RUN_TRAINING and EXISTING_CHECKPOINT:
        saved = json.loads((Path(EXISTING_CHECKPOINT).resolve().parent.parent / 'run_config.json').read_text())
        total = int(saved['training']['batch'])
        if total % devices:
            raise ValueError('Existing checkpoint batch does not divide across the selected GPUs.')
        return total, {'per_gpu': total // devices, 'reason': 'matching existing run'}
    if T4_BATCH_PER_GPU:
        return int(T4_BATCH_PER_GPU) * devices, {'per_gpu': int(T4_BATCH_PER_GPU), 'reason': 'explicit override'}
    from ultralytics.utils.autobatch import check_train_batch_size
    weakest = min(GPU_IDS, key=lambda i: torch.cuda.mem_get_info(i)[0])
    previous_benchmark = torch.backends.cudnn.benchmark
    probe_model = None
    try:
        torch.backends.cudnn.benchmark = False
        probe_model = YOLO(YOLO_MODEL_NAME)
        probe_model.model.to(torch.device(f'cuda:{weakest}'))
        # Bound profiling to [1, 2, 4] per T4, rather than probing large 2048 batches.
        physical = physical_rows(train_records)
        maximum_objects = max(1, max(len(coco.getAnnIds(imgIds=row['image_ids'])) for row in physical))
        import warnings
        try:
            from numpy.exceptions import RankWarning
        except ImportError:  # NumPy 1.x environments
            from numpy import RankWarning
        with torch.cuda.device(weakest):
            # Record, rather than raise: Ultralytics catches Exception internally and
            # can otherwise turn a RankWarning exception into its default batch of 16.
            with warnings.catch_warnings(record=True) as profile_notices:
                warnings.simplefilter('always', RankWarning)
                estimate = check_train_batch_size(
                    probe_model.model, imgsz=IMAGE_SIZE, amp=True,
                    batch=AUTO_BATCH_MEMORY_FRACTION, max_num_obj=maximum_objects,
                    dataset_size=AUTO_BATCH_MAX_PER_GPU)
        fit_warnings = []
        for notice in profile_notices:
            if issubclass(notice.category, RankWarning):
                fit_warnings.append(str(notice.message))
            else:
                # Preserve unrelated CUDA/runtime warnings instead of hiding them.
                warnings.warn_explicit(notice.message, notice.category, notice.filename, notice.lineno)
        per_gpu = int(estimate)
        if fit_warnings or not 1 <= per_gpu <= AUTO_BATCH_MAX_PER_GPU:
            reason = 'ill-conditioned AutoBatch fit' if fit_warnings else 'out-of-range AutoBatch estimate'
            print(f'{reason}; using 1 image per T4 (global batch {devices}).')
            return devices, dict(per_gpu=1, reason=reason, probe_gpu=weakest,
                                 discarded_estimate=per_gpu, profile_warnings=fit_warnings)
        return per_gpu * devices, dict(per_gpu=per_gpu, reason='bounded T4 VRAM profile',
                                       probe_gpu=weakest, memory_fraction=AUTO_BATCH_MEMORY_FRACTION,
                                       max_objects=maximum_objects)
    except (RuntimeError, ValueError, OverflowError) as exc:
        print('T4 batch probe could not select a larger batch; using one image per GPU:', exc)
        return devices, dict(per_gpu=1, reason='probe fallback', error=str(exc))
    finally:
        if probe_model is not None:
            del probe_model
        gc.collect()
        for device in GPU_IDS:
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
        torch.backends.cudnn.benchmark = previous_benchmark


BATCH_SIZE, BATCH_PLAN = choose_dual_t4_batch()
TRAIN_OPTIONS['batch'] = BATCH_SIZE
VALIDATION_BATCH_SIZE = min(2, max(1, BATCH_SIZE // max(1, len(GPU_IDS))))
print('Dual-T4 batch plan:', BATCH_PLAN, '| global batch:', BATCH_SIZE)

VERSIONS = {
    p: version(p)
    for p in [
        "ultralytics",
        "torch",
        "torchvision",
        "numpy",
        "opencv-python",
        "pycocotools",
        "scikit-learn",
        "scipy",
    ]
}
RUN_CONFIG = dict(
    pipeline="dual-t4-precomputed-inputs-d4-pq-v5",
    seed=SEED,
    image_size=IMAGE_SIZE,
    preprocessing_method=PREPROCESSING_METHOD,
    model_input_mode=MODEL_INPUT_MODE,
    preprocessing_source_version=SOURCE_PIPELINE_VERSION,
    preprocessing_runtime_sha256=PREPROCESSING_RUNTIME_SHA256,
    preprocessing=PREPROCESSING_CONFIG,
    target="all_annotators_union_connected_components",
    target_min_area=TARGET_MIN_AREA,
    split_group=SPLIT_GROUP,
    annotation_sha256=file_sha256(ANNOTATION_PATH),
    train=sorted(train_records.file_name.unique()),
    calibration=sorted(calibration_records.file_name.unique()),
    report=sorted(report_records.file_name.unique()),
    model=YOLO_MODEL_NAME,
    training=TRAIN_OPTIONS,
    versions=VERSIONS,
    resources=dict(
        cpu_workers=NUM_WORKERS,
        train_workers_per_rank=YOLO_WORKERS,
        validation_workers=VALIDATION_WORKERS,
        validation_batch_size=VALIDATION_BATCH_SIZE,
        prefetch_factor=TRAIN_PREFETCH_FACTOR,
        cuda_prefetch=CUDA_BATCH_PREFETCH,
        target='Kaggle 2 x Tesla T4', 
        inference_devices=INFERENCE_DEVICES,
    ),
    trainer="prepared_inputs_cuda_prefetch_v2",
    image_cache="temporary_native_bgr_npy",
    selection=dict(
        confidences=CONFIDENCE_CANDIDATES,
        areas=AREA_CANDIDATES,
        tta_views=TTA_VIEWS,
        soft_decoder="native_sigmoid_chunked_v1",
        keep_rules=KEEP_RULES,
        morphology_rules=MORPHOLOGY_RULES,
        disk_clipping=CLIP_TO_DISK_CANDIDATES,
        predict_nms_iou=PREDICT_NMS_IOU,
        pq_match_iou=PQ_MATCH_IOU,
    ),
)


def training_signature(config):
    return {key: value for key, value in config.items() if key != "selection"}


RUN_FINGERPRINT = fingerprint(training_signature(RUN_CONFIG))
EXPERIMENT_DIR = (
    OUTPUT_ROOT
    / f"solar_{MODEL_INPUT_MODE}_dual_t4_cached_{RUN_FINGERPRINT[:10]}"
)
YOLO_DATA_DIR = EXPERIMENT_DIR / "yolo_filament_dataset"
RUNS_DIR = EXPERIMENT_DIR / "yolo_runs"
RUN_NAME = f"{MODEL_TAG}_full_resolution"
DATA_YAML = YOLO_DATA_DIR / "filament.yaml"
PREVIEW_DIR = EXPERIMENT_DIR / "previews"
SUBMISSION_PATH = "a_submission.csv"
save_json(EXPERIMENT_DIR / "run_config.json", RUN_CONFIG)
save_json(EXPERIMENT_DIR / "batch_plan.json", BATCH_PLAN)
for label, frame in [
    ("train", train_records),
    ("calibration", calibration_records),
    ("report", report_records),
]:
    print(
        label,
        ":",
        frame.file_name.nunique(),
        "physical images,",
        len(frame),
        "annotator records,",
        frame.group.nunique(),
        "groups",
    )
    frame.to_csv(EXPERIMENT_DIR / f"{label}_split.csv", index=False)
print("Dataset:", DATA_ROOT, "\nOutputs:", EXPERIMENT_DIR)

## 3. Shared preprocessing and cached-input runtime

The embedded importable module preserves the reference preprocessing functions and
adds the prepared-input loader, persistent/pinned prefetching, and CUDA copy stream.
DDP workers import this same code and settings. It also covers epoch validation and
the final best.pt validation. Keep the runtime and configuration with the weights.

In [ ]:
RUNTIME_DIR = EXPERIMENT_DIR / 'runtime'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_MODULE_NAME = 'solar_streaming_' + PREPROCESSING_RUNTIME_SHA256[:16]
RUNTIME_PATH = RUNTIME_DIR / (RUNTIME_MODULE_NAME + '.py')
if RUNTIME_PATH.exists() and RUNTIME_PATH.read_text(encoding='utf-8') != PREPROCESSING_RUNTIME_SOURCE:
    raise ValueError('Runtime source differs from its fingerprint.')
RUNTIME_PATH.write_text(PREPROCESSING_RUNTIME_SOURCE, encoding='utf-8')
runtime_import_path = str(RUNTIME_DIR.resolve())
if runtime_import_path not in sys.path:
    sys.path.insert(0, runtime_import_path)
importlib.invalidate_caches()
solar_runtime = importlib.import_module(RUNTIME_MODULE_NAME)
StreamingSolarSegmentationTrainer = solar_runtime.StreamingSolarSegmentationTrainer


def preprocess_model_input(image, return_aux=False):
    return solar_runtime.preprocess_model_input(
        image, PREPROCESSING_CONFIG, MODEL_INPUT_MODE, return_aux=return_aux)


def disk_support(shape, disk, margin=1.0):
    if disk is None:
        return np.ones(shape, dtype=bool)
    yy, xx = np.ogrid[:shape[0], :shape[1]]
    return np.hypot(xx - disk['cx'], yy - disk['cy']) <= disk['radius'] * margin


save_json(EXPERIMENT_DIR / 'preprocessing_config.json', dict(
    source_version=SOURCE_PIPELINE_VERSION, source_sha256=PREPROCESSING_RUNTIME_SHA256,
    input_mode=MODEL_INPUT_MODE, parameters=PREPROCESSING_CONFIG,
    tensor_channel_order=['original', 'enhanced', 'candidates'] if MODEL_INPUT_MODE == 'multi_channel'
                         else [MODEL_INPUT_MODE] * 3,
    normalization_scope='per-image on-disk percentiles',
    runtime=str(RUNTIME_PATH.resolve()), image_cache="temporary_native_bgr_npy"))
print('Streaming preprocessing runtime:', RUNTIME_PATH)


def prepared_input_paths(source_path, split):
    stem = Path(source_path).stem
    root = YOLO_DATA_DIR / 'input_cache' / split
    return root / (stem + '.npy'), root / (stem + '.candidates.npy'), root / (stem + '.json')


def atomic_numpy(path, array):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('wb') as stream:
        np.save(stream, array, allow_pickle=False)
    os.replace(temporary, path)


def cache_prepared_input(source_path, split, image=None):
    source_path = Path(source_path)
    path, candidates_path, metadata_path = prepared_input_paths(source_path, split)
    stat = source_path.stat()
    source_signature = [stat.st_size, stat.st_mtime_ns]
    signature = fingerprint(dict(preprocessing=PREPROCESSING_CONFIG, input_mode=MODEL_INPUT_MODE,
                                 runtime=PREPROCESSING_RUNTIME_SHA256))
    if path.is_file() and candidates_path.is_file() and metadata_path.is_file():
        metadata = json.loads(metadata_path.read_text())
        if metadata.get('source_signature') == source_signature and metadata.get('signature') == signature:
            return metadata
    if image is None:
        image = read_gray(source_path)
    processed, auxiliary = preprocess_model_input(image, return_aux=True)
    atomic_numpy(path, processed)
    atomic_numpy(candidates_path, np.packbits(auxiliary['candidates'] > 0, axis=-1))
    metadata = dict(file_name=source_path.name, source_signature=source_signature,
                    source_sha256=file_sha256(source_path), signature=signature,
                    disk=auxiliary['disk'], shape=list(image.shape),
                    preprocessing_seconds=auxiliary['preprocessing_seconds'])
    save_json(metadata_path, metadata)
    return metadata


def model_input_for_path(path, return_aux=False):
    path = Path(path)
    # Train/calibration share filenames with the source aliases; test/report are processed once on demand.
    for split in ('train', 'val'):
        cached_path, candidates_path, metadata_path = prepared_input_paths(path, split)
        if cached_path.is_file() and candidates_path.is_file() and metadata_path.is_file():
            metadata = json.loads(metadata_path.read_text())
            stat = path.stat()
            signature = fingerprint(dict(preprocessing=PREPROCESSING_CONFIG, input_mode=MODEL_INPUT_MODE,
                                         runtime=PREPROCESSING_RUNTIME_SHA256))
            if (metadata.get('file_name') == path.name
                    and metadata.get('source_signature') == [stat.st_size, stat.st_mtime_ns]
                    and metadata.get('signature') == signature):
                processed = solar_runtime.read_cached_bgr(cached_path, PREPROCESSING_CONFIG['expected_shape'])
                if not return_aux:
                    return processed
                packed = np.load(candidates_path, allow_pickle=False)
                candidates = np.unpackbits(packed, axis=-1, count=processed.shape[1]).astype(np.uint8) * 255
                return processed, dict(disk=metadata['disk'], candidates=candidates,
                                       preprocessing_seconds=metadata['preprocessing_seconds'])
    return preprocess_model_input(read_gray(path), return_aux=return_aux)


def validate_prepared_inputs(frame, split):
    signature = fingerprint(dict(preprocessing=PREPROCESSING_CONFIG, input_mode=MODEL_INPUT_MODE,
                                 runtime=PREPROCESSING_RUNTIME_SHA256))
    for name in frame.file_name.unique():
        path, candidates_path, metadata_path = prepared_input_paths(name, split)
        if not (path.is_file() and candidates_path.is_file() and metadata_path.is_file()):
            raise ValueError(f'Incomplete input cache for {name}; set REBUILD_YOLO_DATASET=True.')
        metadata = json.loads(metadata_path.read_text())
        stat = (TRAIN_IMAGE_DIR / name).stat()
        if metadata.get('signature') != signature or metadata.get('source_signature') != [stat.st_size, stat.st_mtime_ns]:
            raise ValueError(f'Stale prepared input for {name}; rebuild the cache before training.')
        if metadata.get('source_sha256') != file_sha256(TRAIN_IMAGE_DIR / name):
            raise ValueError(f'Source content changed for {name}; rebuild the cache before training.')
        mapped = np.load(path, mmap_mode='r', allow_pickle=False)
        try:
            if mapped.shape != (*PREPROCESSING_CONFIG['expected_shape'], 3) or mapped.dtype != np.uint8:
                raise ValueError(f'Invalid prepared input {path}')
        finally:
            if getattr(mapped, '_mmap', None) is not None:
                mapped._mmap.close()


def warm_prepared_input_pages():
    root = YOLO_DATA_DIR / 'input_cache'
    total = directory_bytes(root)
    available = available_memory_bytes()
    warmed = False
    started = time.perf_counter()
    if WARM_OS_PAGE_CACHE and total < max(0, available - RAM_RESERVE_GIB * GIB):
        # File pages are shared between DDP ranks/workers; no full Python RAM image replicas.
        for path in tqdm(sorted(root.rglob('*.npy')), desc='Warming shared input pages'):
            mapped = np.load(path, mmap_mode='r', allow_pickle=False)
            try:
                _ = int(np.sum(mapped.reshape(-1)[::4096], dtype=np.uint64))
            finally:
                if getattr(mapped, '_mmap', None) is not None:
                    mapped._mmap.close()
        warmed = True
    summary = dict(cache_bytes=total, cache_gib=total / GIB, page_cache_warmed=warmed,
                   warmup_seconds=time.perf_counter() - started,
                   preprocessing_in_training_loader=False, storage='uncompressed uint8 BGR NPY',
                   cache_removed_after_success=CLEANUP_INTERMEDIATES)
    save_json(EXPERIMENT_DIR / 'input_cache_summary.json', summary)
    print(f'Prepared input cache: {total/GIB:.2f} GiB; shared pages warmed: {warmed}')
    return summary

## 4. Prepare the temporary model-input cache once

Bounded CPU workers validate originals, prepare lossless native BGR arrays, and
export the existing all-annotator union polygon targets for train/calibration only.
Original file aliases let YOLO verify image/label geometry; the loader reads the
separate prepared arrays. Candidates remain input features, never pseudo-labels.

Atomic array writes and per-image source/configuration metadata prevent partial or
stale cache reuse. Every prepared input is checked before the GPUs start training.
The original channels and source annotations are preserved. Input cache, source
aliases, polygon labels, and auxiliary arrays all live under one owned temporary
workspace and are removed after successful completion. Optional page-cache warming
uses shared file pages rather than one complete RAM copy per rank/worker.

In [ ]:
def merged_annotation_mask(coco_api, image_ids):
    first = coco_api.imgs[image_ids[0]]
    shape = (first["height"], first["width"])
    merged = np.zeros(shape, np.uint8)
    for image_id in image_ids:
        metadata = coco_api.imgs[image_id]
        if (
            metadata["file_name"] != first["file_name"]
            or (metadata["height"], metadata["width"]) != shape
        ):
            raise ValueError("Union records must describe the same physical image.")
        for ann in coco_api.loadAnns(coco_api.getAnnIds(imgIds=[image_id])):
            merged |= coco_api.annToMask(ann).astype(np.uint8)
    return merged


def mask_to_yolo_segments(mask, min_area=TARGET_MIN_AREA):
    h, w = mask.shape
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    segments = []
    for idx in range(1, count):
        if stats[idx, cv2.CC_STAT_AREA] < min_area:
            continue
        x, y, bw, bh, _ = stats[idx]
        crop = (labels[y : y + bh, x : x + bw] == idx).astype(np.uint8)
        contours, _ = cv2.findContours(crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        if len(contour) < 3 or cv2.contourArea(contour) <= 0 or bw < 2 or bh < 2:
            continue
        points = contour[:, 0, :].astype(np.float32) + np.array([x, y], np.float32)
        points /= np.array([w, h], np.float32)
        segments.append(np.clip(points, 0, 1))
    return segments


def directory_bytes(root, suffix=None):
    root = Path(root)
    if not root.is_dir():
        return 0
    return sum(
        p.stat().st_size
        for p in root.rglob("*")
        if p.is_file()
        and not p.is_symlink()
        and (suffix is None or p.suffix.lower() == suffix)
    )


def disk_space_plan(free_gib, experiment_gib=0.0, dataset_gib=0.0,
                    weights_gib=0.0, numpy_cache_gib=0.0, dataset_complete=False):
    count = train_records.file_name.nunique() + calibration_records.file_name.nunique()
    cal_count = calibration_records.file_name.nunique()
    report_count = report_records.file_name.nunique()
    test_count = sum(p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
                     for p in TEST_IMAGE_DIR.iterdir())
    saved_candidates = (NUM_EPOCHS + 2 if SAVE_EVERY_EPOCH else 2) if RUN_TRAINING else 1
    pixels = IMAGE_SIZE ** 2
    map_bytes = (pixels * (len(CONFIDENCE_CANDIDATES) + 1)
                 + len(CONFIDENCE_CANDIDATES) * IMAGE_SIZE * ((IMAGE_SIZE + 7) // 8) + 65536)
    previews = min(report_count, PREDICTION_PREVIEW_COUNT) + min(test_count, PREDICTION_PREVIEW_COUNT)
    if PREDICTION_CACHE_POLICY == 'all':
        peak_images = cal_count * saved_candidates + report_count + test_count + NUM_WORKERS
    elif PREDICTION_CACHE_POLICY == 'bounded':
        peak_images = max(cal_count * min(2, saved_candidates), cal_count + previews) + NUM_WORKERS
    else:
        peak_images = previews + NUM_WORKERS
    # Native BGR inputs plus packed candidate maps; source-copy fallback and polygon labels.
    names = set(train_records.file_name) | set(calibration_records.file_name)
    fallback_bytes = sum((TRAIN_IMAGE_DIR / name).stat().st_size for name in names)
    dataset_growth = 0.0 if dataset_complete else max(0.0, (fallback_bytes + count * (pixels * 3 + IMAGE_SIZE * ((IMAGE_SIZE + 7) // 8) + 262144)) / GIB - dataset_gib)
    weights_growth = max(0.0, max(2.0, saved_candidates * .20) - weights_gib) if RUN_TRAINING else 0.0
    prediction_peak = peak_images * map_bytes / GIB
    budget = max(0.0, min(free_gib, DISK_LIMIT_GIB - experiment_gib))
    return dict(policy=PREDICTION_CACHE_POLICY, budget_gib=budget,
                dataset_growth_gib=dataset_growth, checkpoint_growth_gib=weights_growth,
                prediction_peak_gib=prediction_peak, peak_cached_images=int(peak_images),
                reserve_gib=DISK_RESERVE_GIB,
                required_gib=dataset_growth + weights_growth + prediction_peak + DISK_RESERVE_GIB,
                image_cache_growth_gib=count * pixels * 3 / GIB, yolo_cache=False,
                processed_dataset_saved=True, image_loading='prepared NPY + shared file pages + CUDA prefetch')


def available_memory_bytes():
    available = psutil.virtual_memory().available
    try:
        limit = int(Path('/sys/fs/cgroup/memory.max').read_text())
        used = int(Path('/sys/fs/cgroup/memory.current').read_text())
        available = min(available, limit - used)
    except (OSError, ValueError):
        pass
    return max(0, available)


def resource_preflight():
    available = available_memory_bytes() / GIB
    if available < RAM_RESERVE_GIB + 2:
        raise MemoryError(f'Only {available:.1f} GiB available; free memory before proceeding.')
    manifest_path = YOLO_DATA_DIR / 'manifest.json'
    complete = (not REBUILD_YOLO_DATASET and manifest_path.is_file()
                and json.loads(manifest_path.read_text())['fingerprint'] == RUN_FINGERPRINT)
    plan = disk_space_plan(shutil.disk_usage(EXPERIMENT_DIR).free / GIB,
                           directory_bytes(EXPERIMENT_DIR) / GIB,
                           directory_bytes(YOLO_DATA_DIR) / GIB,
                           directory_bytes(RUNS_DIR, '.pt') / GIB, dataset_complete=complete)
    plan['available_ram_gib'] = available
    save_json(EXPERIMENT_DIR / 'resource_plan.json', plan)
    if plan['budget_gib'] < plan['required_gib']:
        raise OSError(f"Need {plan['required_gib']:.1f} GiB peak headroom, have {plan['budget_gib']:.1f}. See resource_plan.json.")
    print(f"Available RAM {available:.1f} GiB; planned disk peak {plan['required_gib']:.1f} GiB; temporary NPY input cache enabled")
    return False


def checked_generated_directory(path, expected_name):
    path = Path(path)
    resolved = path.resolve()
    if (path.is_symlink() or getattr(path, 'is_junction', lambda: False)()
            or resolved.parent != EXPERIMENT_DIR.resolve() or resolved.name != expected_name
            or resolved == DATA_ROOT.resolve() or DATA_ROOT.resolve() in resolved.parents):
        raise ValueError(f'Refusing to remove an unexpected generated directory: {path}')
    if path.exists():
        for child in path.rglob('*'):
            if child.is_dir() and (child.is_symlink() or getattr(child, 'is_junction', lambda: False)()):
                raise ValueError(f'Refusing to follow a linked directory: {child}')
    return resolved


def remove_generated_dataset():
    root = checked_generated_directory(YOLO_DATA_DIR, 'yolo_filament_dataset')
    marker = root / 'streaming_config.json'
    if root.exists():
        if not marker.is_file() or json.loads(marker.read_text()).get('run_fingerprint') != RUN_FINGERPRINT:
            raise ValueError('Generated dataset ownership marker is missing or stale.')
        shutil.rmtree(root)  # removes aliases themselves, without following source file links


def link_source_image(source, destination):
    source, destination = Path(source).resolve(), Path(destination)
    if not source.is_file():
        raise FileNotFoundError(source)
    if destination.exists() or destination.is_symlink():
        raise FileExistsError(destination)
    try:
        destination.symlink_to(source)
        return 'symlink'
    except OSError:
        try:
            os.link(source, destination)
            return 'hardlink'
        except OSError:
            shutil.copy2(source, destination)
            return 'compressed_source_copy'


def prepare_record(task):
    row, split_name = task

    source_path = TRAIN_IMAGE_DIR / row['file_name']
    # Source validation and label decoding are bounded to this sample.
    image = read_gray(source_path)
    union = merged_annotation_mask(coco, row['image_ids'])
    if union.shape != image.shape:
        raise ValueError('Annotation/image dimension mismatch.')
    stem = Path(row['file_name']).stem
    image_path = YOLO_DATA_DIR / 'images' / split_name / Path(row['file_name']).name
    label_path = YOLO_DATA_DIR / 'labels' / split_name / (stem + '.txt')
    storage = link_source_image(source_path, image_path)
    cache_prepared_input(source_path, split_name, image=image)
    segments = mask_to_yolo_segments(union)
    lines = ["0 " + " ".join(f"{v:.7f}" for v in points.ravel()) for points in segments]
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    return dict(
        file_name=row["file_name"],
        split=split_name,
        annotators=len(row["image_ids"]),
        image_storage=storage,
        prepared_input=True,
        union_pixels=int(union.sum()),
        training_instances=len(segments),
    )


YOLO_CACHE_MODE = resource_preflight()
manifest_path = YOLO_DATA_DIR / "manifest.json"
if REBUILD_YOLO_DATASET and YOLO_DATA_DIR.exists():
    remove_generated_dataset()
if manifest_path.is_file():
    assert json.loads(manifest_path.read_text())["fingerprint"] == RUN_FINGERPRINT
    print("Reusing completed dataset:", YOLO_DATA_DIR)
else:
    if YOLO_DATA_DIR.exists() and any(YOLO_DATA_DIR.iterdir()):
        raise RuntimeError(
            "Incomplete dataset. Set REBUILD_YOLO_DATASET=True to rebuild this generated directory."
        )
    YOLO_DATA_DIR.mkdir(parents=True, exist_ok=True)
    save_json(YOLO_DATA_DIR / 'streaming_config.json', dict(
        run_fingerprint=RUN_FINGERPRINT, preprocessing=PREPROCESSING_CONFIG,
        input_mode=MODEL_INPUT_MODE, runtime_sha256=PREPROCESSING_RUNTIME_SHA256,
        prepared_inputs=True, prefetch_factor=TRAIN_PREFETCH_FACTOR,
        cuda_prefetch=CUDA_BATCH_PREFETCH, validation_workers=VALIDATION_WORKERS,
        validation_batch_size=VALIDATION_BATCH_SIZE))
    tasks = []
    for name, frame in [("train", train_records), ("val", calibration_records)]:
        for sub in ["images", "labels"]:
            (YOLO_DATA_DIR / sub / name).mkdir(parents=True, exist_ok=True)
        tasks.extend((row, name) for row in physical_rows(frame))
    stats = list(
        tqdm(bounded_map(prepare_record, tasks), total=len(tasks), desc="Preparing cached model inputs and targets")
    )
    pd.DataFrame(stats).to_csv(EXPERIMENT_DIR / "target_audit.csv", index=False)
    DATA_YAML.write_text(
        f"path: {json.dumps(YOLO_DATA_DIR.as_posix())}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n",
        encoding="utf-8",
    )
    save_json(manifest_path, {"fingerprint": RUN_FINGERPRINT, "samples": len(tasks)})
for name, frame in [("train", train_records), ("val", calibration_records)]:
    expected = {Path(n).stem for n in frame.file_name.unique()}
    actual_images = {p.stem for p in (YOLO_DATA_DIR / 'images' / name).iterdir()
                     if p.is_file() and p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp'}}
    actual_labels = {p.stem for p in (YOLO_DATA_DIR / 'labels' / name).glob('*.txt')}
    assert actual_images == expected and actual_labels == expected, f'Incomplete {name} aliases/labels.'
print(DATA_YAML.read_text())

for name, frame in [('train', train_records), ('val', calibration_records)]:
    validate_prepared_inputs(frame, name)
input_cache_summary = warm_prepared_input_pages()


In [ ]:
def pick_evenly(items, count):
    items = list(items)
    return (
        [
            items[int(i)]
            for i in np.linspace(0, len(items) - 1, min(count, len(items)), dtype=int)
        ]
        if items
        else []
    )


def show_gray(ax, image):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")


def overlay(ax, image, masks):
    # One RGB display array per panel. A separate RGBA artist per instance
    # would retain gigabytes for a crowded 2048 image.
    rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    for i, mask in enumerate(masks):
        color = np.array(plt.get_cmap("tab20")(i % 20)[:3]) * 255
        rgb[mask] = np.rint(rgb[mask] * 0.55 + color * 0.45).astype(np.uint8)
    ax.imshow(rgb, interpolation="nearest")
    ax.axis("off")


def finish_figure(fig, stage, filename):
    destination = PREVIEW_DIR / stage / f"{Path(filename).stem}.png"
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    if SAVE_PREVIEW_IMAGES:
        fig.savefig(destination, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def preprocessing_target_preview(row):
    image = read_gray(TRAIN_IMAGE_DIR / row['file_name'])
    union = merged_annotation_mask(coco, row['image_ids']).astype(bool)
    stages = solar_runtime.preprocess_image(image, PREPROCESSING_CONFIG)
    processed = preprocess_model_input(image)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, title, array in zip(axes.ravel()[:3], ['Original', 'Enhanced', 'Auxiliary candidates'],
                                [image, stages['enhanced'], stages['candidates'].astype(np.uint8) * 255]):
        show_gray(ax, array)
        ax.set_title(title)
    axes[1, 0].imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
    axes[1, 0].axis('off')
    axes[1, 0].set_title('Actual model input: ' + MODEL_INPUT_MODE)
    overlay(axes[1, 1], image, [union])
    axes[1, 1].set_title(f'Union of {len(row["image_ids"])} annotator records')
    target = np.zeros(image.shape, np.uint8)
    path = YOLO_DATA_DIR / 'labels/train' / (Path(row['file_name']).stem + '.txt')
    for line in path.read_text().splitlines():
        points = np.array(line.split()[1:], np.float32).reshape(-1, 2) * [IMAGE_SIZE, IMAGE_SIZE]
        cv2.fillPoly(target, [np.rint(points).astype(np.int32)], 1)
    overlay(axes[1, 2], stages['enhanced'], [target.astype(bool)])
    intersection = np.logical_and(target, union).sum()
    denominator = np.logical_or(target, union).sum()
    axes[1, 2].set_title(f'YOLO target; union IoU={intersection/max(denominator,1):.4f}')
    fig.suptitle(row['file_name'])
    finish_figure(fig, 'preprocessing_targets', row['file_name'])


for row in pick_evenly(physical_rows(train_records), PREPROCESS_PREVIEW_COUNT):
    preprocessing_target_preview(row)
gc.collect()

## 5. Train both T4s with prepared inputs and overlapped transfers

Preprocessing has finished before this cell. Each rank receives its share of the
memory-profiled batch and two training workers. Persistent loader iterators keep
workers alive, pinned CPU batches queue ahead, and a dedicated CUDA stream copies
the next batch while the current batch computes. The standard trainer normalizes
uint8 values once. Masks and all input channels receive the same geometric transforms.

AMP and fixed-shape cuDNN autotuning are enabled on T4. GPU memory profiling includes
object-count overhead and leaves headroom; the per-GPU cap remains configurable.
The epoch and patience settings are unchanged. Best/last selection and held-out
report scoring remain separate. `input_throughput.csv` records host input waiting
and rank throughput; `gpu_utilization.csv` samples both GPUs once per second.
These are measurements to inspect on Kaggle, not a promised utilization percentage.

Batch/validation montage rendering is disabled during training. The history CSV and
`training_history.png` are produced afterward, outside the GPU training loop.

In [ ]:
def start_gpu_monitor():
    path = EXPERIMENT_DIR / 'gpu_utilization.csv'
    stream = path.open('w', encoding='utf-8')
    stream.write('timestamp,gpu_index,gpu_utilization_percent,memory_used_mib,memory_total_mib\n')
    stream.flush()
    try:
        process = subprocess.Popen(
            ['nvidia-smi', '--query-gpu=timestamp,index,utilization.gpu,memory.used,memory.total',
             '--format=csv,noheader,nounits', '--loop=1'],
            stdout=stream, stderr=subprocess.DEVNULL,
            creationflags=getattr(subprocess, 'CREATE_NO_WINDOW', 0) if os.name == 'nt' else 0)
        return process, stream, path
    except OSError:
        stream.close()
        return None


def stop_gpu_monitor(monitor):
    if monitor is None:
        return
    process, stream, path = monitor
    if process.poll() is None:
        process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)
    stream.close()
    try:
        samples = pd.read_csv(path, skipinitialspace=True)
        for column in ('gpu_utilization_percent', 'memory_used_mib', 'memory_total_mib'):
            samples[column] = pd.to_numeric(samples[column], errors='coerce')
        summary = samples.groupby('gpu_index').agg(
            mean_gpu_utilization_percent=('gpu_utilization_percent', 'mean'),
            peak_memory_mib=('memory_used_mib', 'max'), samples=('timestamp', 'count'))
        summary.to_csv(EXPERIMENT_DIR / 'gpu_utilization_summary.csv')
    except (OSError, ValueError, KeyError):
        pass


training_start = time.perf_counter()
if RUN_TRAINING:
    model = YOLO(YOLO_MODEL_NAME)
    if model.task != "segment":
        raise ValueError(
            f"Expected an instance-segmentation model, got task={model.task}."
        )
    model_description = dict(
        pretrained=YOLO_MODEL_NAME,
        task=model.task,
        head=type(model.model.model[-1]).__name__,
        parameters=sum(p.numel() for p in model.model.parameters()),
    )
    save_json(EXPERIMENT_DIR / "model_description.json", model_description)
    print("Training model:", model_description)
    monitor = start_gpu_monitor()
    try:
        train_result = model.train(
            trainer=StreamingSolarSegmentationTrainer,
            data=str(DATA_YAML),
            cache=False,
            project=str(RUNS_DIR),
            name=RUN_NAME,
            exist_ok=False,
            **TRAIN_OPTIONS,
        )
    finally:
        stop_gpu_monitor(monitor)
    run_dir = Path(model.trainer.save_dir)
    checkpoint_candidates = [
        run_dir / "weights" / name for name in ["best.pt", "last.pt"]
    ]
    assert all(p.is_file() for p in checkpoint_candidates)
    if SAVE_EVERY_EPOCH:
        checkpoint_candidates += sorted((run_dir / "weights").glob("epoch*.pt"))
    unique_checkpoints = {}
    for candidate in checkpoint_candidates:
        unique_checkpoints.setdefault(file_sha256(candidate), candidate)
    checkpoint_candidates = list(unique_checkpoints.values())
    save_json(run_dir / "run_config.json", RUN_CONFIG)
    del train_result, model
else:
    if not EXISTING_CHECKPOINT:
        raise ValueError("Set EXISTING_CHECKPOINT when RUN_TRAINING=False.")
    checkpoint_candidates = [Path(EXISTING_CHECKPOINT).resolve()]
    run_dir = checkpoint_candidates[0].parent.parent
    previous = json.loads((run_dir / "run_config.json").read_text(encoding="utf-8"))
    if training_signature(previous) != training_signature(RUN_CONFIG):
        raise ValueError(
            "Checkpoint transform/targets/split/config differs from this run."
        )
training_seconds = time.perf_counter() - training_start
gc.collect()
if GPU_IDS:
    torch.cuda.empty_cache()
print("Checkpoint candidates:", checkpoint_candidates)
history_path = run_dir / "results.csv"
if history_path.is_file():
    history = pd.read_csv(history_path)
    history.columns = history.columns.str.strip()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for column in ["train/seg_loss", "val/seg_loss"]:
        if column in history:
            axes[0].plot(np.arange(1, len(history) + 1), history[column], label=column)
    for column in ["metrics/mAP50(M)", "metrics/mAP50-95(M)"]:
        if column in history:
            axes[1].plot(np.arange(1, len(history) + 1), history[column], label=column)
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.2)
        if ax.lines:
            ax.legend()
    axes[0].set_title("Mask loss")
    axes[1].set_title("Calibration mask mAP (not PQ)")
    plt.tight_layout()
    fig.savefig(run_dir / "training_history.png", dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## Check Training History


In [ ]:
yolo_visual_groups = {
    "Overall Performance & Loss Tracking": ["training_history.png", "results.png"],
    "Bounding Box Localization Metrics": [
        "BoxF1_curve.png",
        "BoxPR_curve.png",
        "BoxP_curve.png",
        "BoxR_curve.png",
    ],
    "Instance Segmentation Mask Metrics": [
        "MaskF1_curve.png",
        "MaskPR_curve.png",
        "MaskP_curve.png",
        "MaskR_curve.png",
    ],
    "Confusion Matrices": ["confusion_matrix.png", "confusion_matrix_normalized.png"],
    "Labels": ["labels.jpg"],
    "Training Dataset Batches & Anchors": [
        "train_batch0.jpg",
        "train_batch1.jpg",
        "train_batch2.jpg",
    ],
    "Validation Predictions vs Ground Truth": [
        "val_batch0_labels.jpg",
        "val_batch0_pred.jpg",
        "val_batch1_labels.jpg",
        "val_batch1_pred.jpg",
        "val_batch2_labels.jpg",
        "val_batch2_pred.jpg",
    ],
}


display_settings = {
    "Overall Performance & Loss Tracking": {
        "cols": 1,
        "col_width": 16,
        "row_height": 9,
    },
    "Bounding Box Localization Metrics": {
        "cols": 2,
        "col_width": 8,
        "row_height": 6,
    },
    "Instance Segmentation Mask Metrics": {
        "cols": 2,
        "col_width": 8,
        "row_height": 6,
    },
    "Confusion Matrices": {
        "cols": 1,
        "col_width": 10,
        "row_height": 8,
    },
    "Labels": {
        "cols": 1,
        "col_width": 12,
        "row_height": 8,
    },
    "Training Dataset Batches & Anchors": {
        "cols": 3,
        "col_width": 6,
        "row_height": 5,
    },
    "Validation Predictions vs Ground Truth": {
        "cols": 2,
        "col_width": 6,
        "row_height": 5,
    },
}


print(f"Loading generated artifact visuals from: {run_dir}\n" + "=" * 70)


for group_name, plot_files in yolo_visual_groups.items():
    valid_files = [f for f in plot_files if (run_dir / f).is_file()]

    if not valid_files:
        continue

    print(f"\n▶ Displaying Group: {group_name}")
    print("-" * 70)

    settings = display_settings[group_name]

    cols = settings["cols"]
    col_width = settings["col_width"]
    row_height = settings["row_height"]

    num_plots = len(valid_files)
    num_rows = (num_plots + cols - 1) // cols

    fig, axes = plt.subplots(
        num_rows,
        cols,
        figsize=(
            col_width * cols,
            row_height * num_rows,
        ),
    )

    axes = np.atleast_1d(axes).flatten()

    for idx, plot_name in enumerate(valid_files):
        img = cv2.imread(str(run_dir / plot_name))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        axes[idx].imshow(img)
        axes[idx].set_title(
            plot_name,
            fontsize=11,
            fontweight="bold",
            pad=8,
        )
        axes[idx].axis("off")

    for idx in range(num_plots, len(axes)):
        axes[idx].axis("off")

    fig.tight_layout()
    plt.show()

## 6. Shared streaming inference and component extraction

Calibration reuses its prepared inputs. Every report/test source is preprocessed once into the same BGR
feature array used by the training loader. D4 transforms move its three channels
together and inverse-transform predicted probabilities. Color transforms are disabled.
The candidate map is available only as the model's feature channel and an optional
YOLO-supported morphology guide; it never becomes an unrestricted prediction.

Prediction probability maps are transient compressed arrays. The default stream
policy deletes each calibration map after all rules are scored, retaining only
bounded report/test previews until display. The selected checkpoint/configuration,
metrics, final masks, and audits survive cleanup.

In [ ]:
def encode_mask(mask):
    return mask_utils.encode(np.asfortranarray(mask, dtype=np.uint8))


def json_rle(rle):
    return {
        "size": list(rle["size"]),
        "counts": (
            rle["counts"].decode("ascii")
            if isinstance(rle["counts"], bytes)
            else rle["counts"]
        ),
    }


def native_rle(rle):
    return {
        "size": list(rle["size"]),
        "counts": (
            rle["counts"].encode("ascii")
            if isinstance(rle["counts"], str)
            else rle["counts"]
        ),
    }


def decode_mask(rle):
    return mask_utils.decode(native_rle(rle)).astype(bool)


D4 = [
    (0, False),
    (0, True),
    (2, False),
    (2, True),
    (1, False),
    (1, True),
    (3, False),
    (3, True),
]


def d4_transform(array, k, flip):
    out = np.flip(array, axis=1) if flip else array
    return np.ascontiguousarray(np.rot90(out, k, axes=(0, 1)))


def d4_inverse_maps(array, k, flip):
    out = np.rot90(array, -k, axes=(-2, -1))
    return np.ascontiguousarray(np.flip(out, axis=-1) if flip else out)


class NativeProbabilityPredictor(SegmentationPredictor):
    def construct_result(self, pred, img, orig_img, img_path, proto):
        shape = orig_img.shape[:2]
        maps = torch.zeros(
            (len(CONFIDENCE_CANDIDATES), *shape),
            dtype=torch.float32,
            device=proto.device,
        )
        if len(pred):
            pred[:, :4] = ops.scale_boxes(img.shape[2:], pred[:, :4], orig_img.shape)
            channels, mh, mw = proto.shape
            # Sigmoid after native logit interpolation, then zero OUTSIDE each box.
            for start in range(0, len(pred), MASK_DECODE_CHUNK):
                detections = pred[start : start + MASK_DECODE_CHUNK]
                logits = (
                    detections[:, 6:].float() @ proto.float().view(channels, -1)
                ).view(-1, mh, mw)
                probs = ops.scale_masks(logits[None], shape)[0].sigmoid()
                probs = ops.crop_mask(probs, detections[:, :4])
                for index, threshold in enumerate(CONFIDENCE_CANDIDATES):
                    keep = detections[:, 4] >= threshold
                    if keep.any():
                        torch.maximum(maps[index], probs[keep].amax(0), out=maps[index])
        result = Results(
            orig_img, path=img_path, names=self.model.names, boxes=pred[:, :6]
        )
        result.single_binary = maps.gt(0.5).to(torch.uint8).cpu().numpy()
        result.probability_maps = (
            maps.mul(255).round().clamp_(0, 255).to(torch.uint8).cpu().numpy()
        )
        return result


class InferenceEngine:
    """One independently initialized YOLO/predictor per device; queue grants exclusive ownership."""

    def __init__(self, checkpoint):
        self.lanes = queue.Queue()
        self.models = []
        for name in INFERENCE_DEVICES:
            device = torch.device(
                name
            )  # Avoid changing CUDA_VISIBLE_DEVICES per worker.
            model = YOLO(str(checkpoint))
            predictor = NativeProbabilityPredictor(
                overrides=dict(
                    task="segment",
                    mode="predict",
                    imgsz=IMAGE_SIZE,
                    batch=1,
                    device=device,
                    quantize=16 if device.type == "cuda" else None,
                    conf=min(CONFIDENCE_CANDIDATES),
                    max_det=MAX_DETECTIONS,
                    retina_masks=True,
                    verbose=False,
                    save=False,
                )
            )
            # Initialize/fuse serially: no first-call global setup race across GPU threads.
            predictor.setup_model(model=model.model, verbose=False)
            model.predictor = predictor
            self.models.append(model)
            self.lanes.put((model, device))

    def predict_maps(self, processed):
        model, device = self.lanes.get()
        try:
            total = np.zeros((len(CONFIDENCE_CANDIDATES), *processed.shape[:2]), np.uint16)
            single_binary = None
            for view_index, (k, flip) in enumerate(D4[:TTA_VIEWS]):
                view = d4_transform(processed, k, flip)
                result = model.predict(
                    source=view,
                    imgsz=IMAGE_SIZE,
                    conf=min(CONFIDENCE_CANDIDATES),
                    iou=PREDICT_NMS_IOU,
                    retina_masks=True,
                    max_det=MAX_DETECTIONS,
                    device=device,
                    quantize=16 if device.type == "cuda" else None,
                    batch=1,
                    nms=None,
                    augment=False,
                    save=False,
                    verbose=False,
                )[0]
                total += d4_inverse_maps(result.probability_maps, k, flip)
                if view_index == 0:
                    single_binary = result.single_binary
                del result
            probability = np.rint(total.astype(np.float32) / TTA_VIEWS).astype(np.uint8)
            return probability, single_binary
        finally:
            self.lanes.put((model, device))

    def close(self):
        while not self.lanes.empty():
            self.lanes.get_nowait()
        self.models.clear()
        gc.collect()
        for device in GPU_IDS:
            with torch.cuda.device(device):
                torch.cuda.empty_cache()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()


def current_inference_config():
    return dict(
        tta_views=TTA_VIEWS,
        confidences=list(CONFIDENCE_CANDIDATES),
        decoder="native_sigmoid_chunked_v1",
        image_size=IMAGE_SIZE,
        preprocessing_method=PREPROCESSING_METHOD,
        preprocessing=dict(PREPROCESSING_CONFIG),
        input_mode=MODEL_INPUT_MODE,
        preprocessing_runtime_sha256=PREPROCESSING_RUNTIME_SHA256,
        predict_nms_iou=PREDICT_NMS_IOU,
        max_det=MAX_DETECTIONS,
    )


def inference_cache_dir(checkpoint):
    signature = dict(
        weights=file_sha256(checkpoint),
        run=RUN_FINGERPRINT,
        inference=current_inference_config(),
    )
    root = EXPERIMENT_DIR / "prediction_cache" / fingerprint(signature)[:16]
    root.mkdir(parents=True, exist_ok=True)
    return root


def checked_prediction_cache_dir(cache_dir):
    path = Path(cache_dir)
    expected = EXPERIMENT_DIR / "prediction_cache"
    resolved = path.resolve()
    if (
        path.is_symlink()
        or expected.is_symlink()
        or resolved.parent != expected.resolve()
        or re.fullmatch(r"[0-9a-f]{16}", resolved.name) is None
    ):
        raise ValueError(
            f"Refusing to prune outside this experiment's generated prediction cache: {path}"
        )
    return resolved


def delete_prediction_source(cache_dir, stem):
    root = checked_prediction_cache_dir(cache_dir)
    if Path(stem).name != stem or stem in {"", ".", ".."}:
        raise ValueError("Invalid prediction-cache stem.")
    for suffix in [".npz", ".json", ".npz.tmp", ".json.tmp"]:
        path = root / (stem + suffix)
        if path.is_symlink():
            raise ValueError(f"Refusing to prune a linked cache file: {path}")
        path.unlink(missing_ok=True)


def prune_prediction_cache(cache_dir, keep_stems=()):
    root = checked_prediction_cache_dir(cache_dir)
    keep = set(keep_stems)
    removed = 0
    for path in list(root.iterdir()):
        # Only our flat per-image NPZ/JSON artifacts; never weights or datasets.
        suffix = next(
            (
                s
                for s in [".npz.tmp", ".json.tmp", ".npz", ".json"]
                if path.name.endswith(s)
            ),
            None,
        )
        if suffix is None or not path.is_file():
            continue
        if path.name[: -len(suffix)] not in keep:
            if path.is_symlink():
                raise ValueError(f"Refusing to prune a linked cache file: {path}")
            removed += path.stat().st_size
            path.unlink()
    return removed


def cached_predict(engine, path, cache_dir):
    path = Path(path)
    destination = cache_dir / f"{path.stem}.json"
    map_path = cache_dir / f"{path.stem}.npz"
    stat = path.stat()
    source_signature = [str(path.resolve()), stat.st_size, stat.st_mtime_ns]
    if destination.is_file() and map_path.is_file():
        cached = json.loads(destination.read_text(encoding="utf-8"))
        if cached["source"] == source_signature and cached.get(
            "inference_signature"
        ) == fingerprint(current_inference_config()):
            cached["map_path"] = str(map_path)
            return cached
    if engine is None:
        raise RuntimeError(
            f"Cache missing/stale for {path.name}; rerun its inference stage."
        )
    processed, auxiliary = model_input_for_path(path, return_aux=True)
    probability, single_binary = engine.predict_maps(processed)
    disk = auxiliary["disk"]
    salience = auxiliary["candidates"]
    temporary = map_path.with_suffix(".npz.tmp")
    with temporary.open("wb") as stream:
        np.savez_compressed(
            stream,
            probability=probability,
            single_binary=np.packbits(single_binary, axis=-1),
            salience=salience,
        )
    os.replace(temporary, map_path)
    cached = dict(
        source=source_signature,
        file_name=path.name,
        disk=disk,
        map_path=str(map_path),
        confidences=CONFIDENCE_CANDIDATES,
        tta_views=TTA_VIEWS,
        preprocessing_seconds=auxiliary["preprocessing_seconds"],
        inference_signature=fingerprint(current_inference_config()),
    )
    save_json(destination, cached)  # Publish metadata only after the maps are complete.
    return cached


def load_maps(cached):
    with np.load(cached["map_path"], allow_pickle=False) as data:
        maps = {key: data[key] for key in ["probability", "single_binary", "salience"]}
    maps["single_binary"] = np.unpackbits(
        maps["single_binary"], axis=-1, count=maps["probability"].shape[-1]
    )
    return maps


def component_base_key(cfg):
    return (
        cfg["confidence"],
        cfg["mode"],
        cfg["clip_to_disk"],
        cfg.get("t_lo", 0.5),
        cfg.get("bridge_px", 0),
    )


def component_candidates(cached, maps, cfg):
    index = cached["confidences"].index(cfg["confidence"])
    p = maps["probability"][index]
    if cfg["mode"] == "union_cc":
        binary = maps["single_binary"][index].copy()
    else:
        binary = (p > int(round(cfg["t_lo"] * 255))).astype(np.uint8)
        if cfg["mode"] == "hysteresis_morph" and cached["disk"] is not None:
            width = cfg["bridge_px"]
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (width, width))
            near = cv2.dilate(binary, kernel).astype(bool)
            permitted = p > int(round(0.15 * 255))
            extension = near & permitted & (maps["salience"] >= 48)  # shared auxiliary candidate map; no morphology-only predictions
            binary |= extension.astype(np.uint8)
            binary = cv2.morphologyEx(
                binary, cv2.MORPH_CLOSE, kernel
            ) & permitted.astype(np.uint8)
    if cfg["clip_to_disk"]:
        binary &= disk_support(binary.shape, cached["disk"]).astype(np.uint8)
    count, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    output = []
    for label in range(1, count):
        x, y, w, h, area = (int(v) for v in stats[label])
        roi = labels[y : y + h, x : x + w] == label
        values = p[y : y + h, x : x + w][roi]
        full = np.zeros(binary.shape, np.uint8)
        full[y : y + h, x : x + w] = roi
        output.append(
            dict(
                rle=encode_mask(full),
                area=area,
                meanp=float(values.mean()) / 255,
                peak=int(values.max()),
                x=x,
                y=y,
            )
        )
    return output


def keep_components(candidates, cfg):
    kept = [c for c in candidates if c["area"] >= max(1, cfg["min_area"])]
    if cfg["mode"] != "union_cc":
        kept = [
            c
            for c in kept
            if c["peak"] > int(round(cfg["t_hi"] * 255))
            and c["meanp"] >= cfg["min_meanp"]
        ]
    return [c["rle"] for c in sorted(kept, key=lambda c: (-c["area"], c["y"], c["x"]))]


def canonical_postprocessing_config(config):
    cfg = dict(config)
    if cfg.get("mode") == "instances":
        cfg["mode"] = "union_cc"  # Compatibility with old disjoint-export repair.
    if cfg["mode"] not in {"union_cc", "hysteresis", "hysteresis_morph"}:
        raise ValueError(f"Unknown postprocessing mode: {cfg['mode']}")
    return cfg


def final_prediction(cached, config):
    cfg = canonical_postprocessing_config(config)
    return keep_components(component_candidates(cached, load_maps(cached), cfg), cfg)


POSTPROCESS_GRID = []
for confidence in CONFIDENCE_CANDIDATES:
    for clip in CLIP_TO_DISK_CANDIDATES:
        base = dict(confidence=confidence, clip_to_disk=clip)
        POSTPROCESS_GRID.extend(
            dict(**base, mode="union_cc", min_area=area) for area in AREA_CANDIDATES
        )
        POSTPROCESS_GRID.extend(
            dict(**base, mode="hysteresis", **rule) for rule in KEEP_RULES
        )
        POSTPROCESS_GRID.extend(
            dict(**base, mode="hysteresis_morph", **rule) for rule in MORPHOLOGY_RULES
        )
print(
    f"Calibration candidates: {len(POSTPROCESS_GRID)} per checkpoint; GPUs: {INFERENCE_DEVICES}; CPU pool: {NUM_WORKERS}"
)

In [ ]:
def target_rles(image_id):
    rles = [coco.annToRLE(a) for a in coco.loadAnns(coco.getAnnIds(imgIds=[image_id]))]
    return [r for r in rles if int(mask_utils.area(r)) > 0]


def score_rles(predictions, targets):
    n_pred, n_gt = len(predictions), len(targets)
    overlaps = (
        mask_utils.iou(predictions, targets, [0] * n_gt).T
        if n_pred and n_gt
        else np.zeros((n_gt, n_pred), np.float64)
    )
    hits = overlaps > PQ_MATCH_IOU
    # Exactly the organizers' published self-evaluation logic, including overlap cases.
    tp = int(hits.sum())
    fp = int((hits.sum(axis=0) == 0).sum())
    fn = int((hits.sum(axis=1) == 0).sum())
    iou_sum = float(overlaps[hits].sum())
    den = tp + 0.5 * (fp + fn)
    pred_union = mask_utils.merge(predictions) if predictions else None
    gt_union = mask_utils.merge(targets) if targets else None
    pred_area = int(mask_utils.area(pred_union)) if pred_union is not None else 0
    gt_area = int(mask_utils.area(gt_union)) if gt_union is not None else 0
    intersection = (
        int(mask_utils.area(mask_utils.merge([pred_union, gt_union], intersect=True)))
        if pred_area and gt_area
        else 0
    )
    return dict(
        tp=tp,
        fp=fp,
        fn=fn,
        matched_iou_sum=iou_sum,
        pq=iou_sum / den if den else 0.0,
        one_to_many=int((hits.sum(axis=1) > 1).sum()),
        many_to_one=int((hits.sum(axis=0) > 1).sum()),
        n_pred=n_pred,
        n_gt=n_gt,
        pixel_dice=(
            2 * intersection / (pred_area + gt_area) if pred_area + gt_area else 1.0
        ),
        pixel_iou=(
            intersection / (pred_area + gt_area - intersection)
            if pred_area + gt_area
            else 1.0
        ),
    )


def pooled_metrics(rows):
    frame = pd.DataFrame(rows)
    tp, fp, fn = (int(frame[c].sum()) for c in ["tp", "fp", "fn"])
    iou_sum = float(frame.matched_iou_sum.sum())
    denominator = tp + 0.5 * (fp + fn)
    return dict(
        PQ=iou_sum / denominator if denominator else 0.0,
        SQ=iou_sum / tp if tp else 0.0,
        RQ=tp / denominator if denominator else 0.0,
        TP=tp,
        FP=fp,
        FN=fn,
        mean_Dice=float(frame.pixel_dice.mean()),
        mean_IoU=float(frame.pixel_iou.mean()),
        annotator_records=len(frame),
        one_to_many=int(frame.one_to_many.sum()),
        many_to_one=int(frame.many_to_one.sum()),
    )


def score_one_source(cached, image_ids, configs):
    targets = {image_id: target_rles(image_id) for image_id in image_ids}
    maps = load_maps(cached)
    output, current_key, candidates = [], None, None
    ordered = sorted(enumerate(configs), key=lambda item: component_base_key(item[1]))
    for config_idx, cfg in ordered:
        key = component_base_key(cfg)
        if key != current_key:
            candidates = component_candidates(cached, maps, cfg)
            current_key = key
        prediction = keep_components(candidates, cfg)
        for image_id, gt in targets.items():
            output.append(
                dict(
                    config_idx=config_idx,
                    image_id=image_id,
                    file_name=cached["file_name"],
                    **score_rles(prediction, gt)
                )
            )
    return output


def evaluate_partition(
    engine, frame, configs, cache_dir, description, retain_cache_stems=None
):
    physical = physical_rows(frame)

    def task(row):
        cached = cached_predict(engine, TRAIN_IMAGE_DIR / row["file_name"], cache_dir)
        scores = score_one_source(cached, row["image_ids"], configs)
        if (
            retain_cache_stems is not None
            and Path(row["file_name"]).stem not in retain_cache_stems
        ):
            delete_prediction_source(cache_dir, Path(row["file_name"]).stem)
        return scores

    rows = []
    for result in tqdm(
        bounded_map(task, physical), total=len(physical), desc=description
    ):
        rows.extend(result)
    per_record = pd.DataFrame(rows)
    summaries = [
        dict(
            config_idx=i,
            **cfg,
            **pooled_metrics(per_record[per_record.config_idx == i].to_dict("records"))
        )
        for i, cfg in enumerate(configs)
    ]
    return per_record, pd.DataFrame(summaries)

## 7. Select on calibration, then freeze and report

The selected score on calibration is optimistic because it is used for selection. **Report PQ** is the separate estimate. Use these report scores sparingly when comparing bilateral/solar/raw/CLAHE; repeated tuning against the report also turns it into a selection set. Large leaderboard gains are not guaranteed. Per-annotator counts are pooled, rather than averaging per-image PQ.

Checkpoint candidates run sequentially. Under the default bounded policy, calibration keeps the current winner's maps and the candidate being evaluated, then prunes the losing cache. Strictly higher PQ replaces the incumbent, preserving the same stable tie rule as the summary table. Scores for every candidate remain in CSV and weight files are retained. Report images are scored in a four-task stream; only the selected preview maps are kept.


In [ ]:
validation_start = time.perf_counter()
all_summaries = []
best_cache_dir, best_cache_pq = None, -float("inf")
cache_retention_log = []
for checkpoint in checkpoint_candidates:
    cache_dir = inference_cache_dir(checkpoint)
    with InferenceEngine(checkpoint) as engine:
        per_record, summary = evaluate_partition(
            engine,
            calibration_records,
            POSTPROCESS_GRID,
            cache_dir,
            f"Calibration {checkpoint.name}",
            retain_cache_stems=set() if PREDICTION_CACHE_POLICY == "stream" else None,
        )
    summary["checkpoint"] = str(checkpoint.resolve())
    summary["weights_sha256"] = file_sha256(checkpoint)
    all_summaries.append(summary)
    per_record.to_csv(
        EXPERIMENT_DIR / f"calibration_{checkpoint.stem}_per_record.csv", index=False
    )
    candidate_pq = float(summary.PQ.max())
    removed_bytes = 0
    if PREDICTION_CACHE_POLICY in {"stream", "bounded"}:
        if candidate_pq > best_cache_pq:
            if best_cache_dir is not None and best_cache_dir != cache_dir:
                removed_bytes = prune_prediction_cache(best_cache_dir)
            best_cache_dir, best_cache_pq = cache_dir, candidate_pq
        elif best_cache_dir != cache_dir:
            removed_bytes = prune_prediction_cache(cache_dir)
    cache_retention_log.append(
        dict(
            checkpoint=str(checkpoint),
            PQ=candidate_pq,
            removed_cache_bytes=removed_bytes,
        )
    )
    save_json(EXPERIMENT_DIR / "cache_retention_log.json", cache_retention_log)
    gc.collect()
    if GPU_IDS:
        torch.cuda.empty_cache()
calibration_summary = pd.concat(all_summaries, ignore_index=True)
calibration_summary = calibration_summary.sort_values(
    "PQ", ascending=False, kind="stable"
).reset_index(drop=True)
calibration_summary.to_csv(EXPERIMENT_DIR / "calibration_grid.csv", index=False)
winner = calibration_summary.iloc[0]
SELECTED_CONFIG = canonical_postprocessing_config(
    POSTPROCESS_GRID[int(winner.config_idx)]
)
SELECTED_CHECKPOINT = Path(winner.checkpoint)
SELECTED_CACHE_DIR = inference_cache_dir(SELECTED_CHECKPOINT)
FROZEN_SELECTION = dict(
    model=YOLO_MODEL_NAME,
    checkpoint=str(SELECTED_CHECKPOINT),
    weights_sha256=winner.weights_sha256,
    postprocessing=SELECTED_CONFIG,
    inference=current_inference_config(),
    mask_postprocessing_version="solar_streaming_candidate_guidance_nonoverlap_v4",
    calibration_PQ=float(winner.PQ),
    run_fingerprint=RUN_FINGERPRINT,
)
save_json(EXPERIMENT_DIR / "selected_pipeline.json", FROZEN_SELECTION)
display(calibration_summary.head(12))
mode_comparison = (
    calibration_summary.sort_values("PQ", ascending=False)
    .groupby("mode", sort=False)
    .head(1)
)
mode_comparison.to_csv(EXPERIMENT_DIR / "calibration_mode_comparison.csv", index=False)
display(
    mode_comparison[["checkpoint", "mode", "confidence", "min_area", "PQ", "FP", "FN"]]
)
print("Frozen selection:", FROZEN_SELECTION)

report_preview_stems = {
    Path(row["file_name"]).stem
    for row in pick_evenly(physical_rows(report_records), PREDICTION_PREVIEW_COUNT)
}
with InferenceEngine(SELECTED_CHECKPOINT) as engine:
    report_per_record, report_summary = evaluate_partition(
        engine,
        report_records,
        [SELECTED_CONFIG],
        SELECTED_CACHE_DIR,
        "Held-out report",
        retain_cache_stems=(
            report_preview_stems if PREDICTION_CACHE_POLICY != "all" else None
        ),
    )
report_per_record.to_csv(EXPERIMENT_DIR / "report_per_annotator.csv", index=False)
report_summary.to_csv(EXPERIMENT_DIR / "report_metrics.csv", index=False)
validation_seconds = time.perf_counter() - validation_start
display(report_summary)
save_json(
    EXPERIMENT_DIR / "timings.json",
    dict(training_seconds=training_seconds, validation_seconds=validation_seconds),
)
fig, ax = plt.subplots(figsize=(8, 4))
for (ckpt, mode, clip), frame in calibration_summary.groupby(
    ["checkpoint", "mode", "clip_to_disk"]
):
    best_per_conf = frame.groupby("confidence").PQ.max()
    ax.plot(
        best_per_conf.index,
        best_per_conf.values,
        marker=".",
        label=f"{Path(ckpt).name}, {mode}, clip={clip}",
    )
ax.set(
    xlabel="Confidence",
    ylabel="Calibration PQ",
    title="Best keep rule per curve; calibration selection scores",
)
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)

## 8. Ten validation prediction previews

Evenly spaced distinct report images, including empty predictions. The diagnostic overlay shows the single-view union; the middle bottom panel shows the selected final masks. Peak/mean filters and TTA can change both count and boundaries. Metrics score every annotator separately; one annotator is displayed for readability. These cells read already-computed caches.


In [ ]:
def prediction_preview(path, cached, final_rles, stage, gt_rles=None):
    image = read_gray(path)
    processed = model_input_for_path(path)
    maps = load_maps(cached)
    raw = [maps["single_binary"][0].astype(bool)]
    final = [decode_mask(r) for r in final_rles]
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.ravel()
    show_gray(axes[0], image)
    axes[0].set_title("Original 2048 image")
    axes[1].imshow(cv2.cvtColor(processed, cv2.COLOR_BGR2RGB))
    axes[1].axis("off")
    axes[1].set_title(f"Model input: {MODEL_INPUT_MODE}")
    overlay(axes[2], image, raw)
    axes[2].set_title(f"Single-view union (conf ≥ {min(CONFIDENCE_CANDIDATES):.2f})")
    if gt_rles is not None:
        overlay(axes[3], image, [decode_mask(r) for r in gt_rles])
        axes[3].set_title("GT: one annotator (display only)")
    else:
        merged = np.zeros(image.shape, np.uint8)
        for mask in final:
            merged |= mask.astype(np.uint8)
        axes[3].imshow(merged, cmap="gray", vmin=0, vmax=1)
        axes[3].axis("off")
        axes[3].set_title("Final mask decoded from submission CSV")
    overlay(axes[4], image, final)
    axes[4].set_title(f"Final predicted instances: {len(final)}")
    if final_rles:
        x, y, w, h = mask_utils.toBbox(native_rle(final_rles[0])).astype(int)
        pad = max(40, int(max(w, h) * 0.20))
        x0, x1 = max(0, x - pad), min(IMAGE_SIZE, x + w + pad)
        y0, y1 = max(0, y - pad), min(IMAGE_SIZE, y + h + pad)
        overlay(axes[5], image[y0:y1, x0:x1], [m[y0:y1, x0:x1] for m in final])
        axes[5].set_title("Largest instance: native-pixel detail")
    else:
        show_gray(axes[5], image)
        axes[5].set_title("No predicted filaments")
    fig.suptitle(f"{Path(path).name} | {SELECTED_CONFIG}", fontsize=11)
    finish_figure(fig, stage, Path(path).name)


for row in pick_evenly(physical_rows(report_records), PREDICTION_PREVIEW_COUNT):
    path = TRAIN_IMAGE_DIR / row["file_name"]
    cached = cached_predict(None, path, SELECTED_CACHE_DIR)
    prediction_preview(
        path,
        cached,
        final_prediction(cached, SELECTED_CONFIG),
        "validation_predictions",
        target_rles(row["image_ids"][0]),
    )

## 9. Competition submission — original 0.36 CSV contract

The selected checkpoint, input transform, TTA and extraction rule are frozen before test inference. Both GPUs process test images through the same bounded four-worker path used for validation. CSV rows are written in sorted image order, with per-image IDs starting at one and masks sorted by area/top/left. The exact original `mask_to_rle`, header `filament_id,segmentation_rle`, default `csv.writer`, and native 2048 mask shape are retained.

Zero-prediction images produce no placeholder row but are recorded in the inference log. A temporary CSV is fully decoded, round-tripped, checked for unique IDs and overlapping masks, then atomically renamed. The audit verifies the output format, not competition performance.


In [ ]:
def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return encoded["counts"].decode("utf-8")


def get_test_image_paths():
    paths = [
        path
        for path in TEST_IMAGE_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
    ]
    return sorted(paths, key=lambda path: path.name)


def test_image_paths():
    return get_test_image_paths()


def submission_rows_for_source(path_stem, rles):
    # The masks come from the same selected postprocessing used in validation.
    # Encode each native mask with the exact original encoder and ID loop.
    components = (decode_mask(rle) for rle in rles)
    rows = []
    for filament_counter, component in enumerate(components, start=1):
        if component.shape != (2048, 2048):
            raise ValueError("Submission masks must be exactly 2048 x 2048.")
        rows.append((f"{path_stem}_{filament_counter}", mask_to_rle(component)))
    return rows


def create_submission_parallel(output_path=SUBMISSION_PATH):
    # Accept both strings and Path objects, as open() did in the original.
    output_path = Path(output_path)
    if FROZEN_SELECTION.get("inference") != current_inference_config():
        raise ValueError(
            "Inference settings changed after selection; rerun calibration before export."
        )
    paths = get_test_image_paths()
    if not paths:
        raise FileNotFoundError(f"No .jpg/.jpeg images found in {TEST_IMAGE_DIR}")
    if len({path.stem for path in paths}) != len(paths):
        raise ValueError("Test images must have unique filename stems.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    processed = {}
    seen_filament_ids = set()
    preview_stems = {p.stem for p in pick_evenly(paths, PREDICTION_PREVIEW_COUNT)}
    start_time = time.perf_counter()
    export_config = canonical_postprocessing_config(SELECTED_CONFIG)
    export_selection = dict(FROZEN_SELECTION)
    export_selection["postprocessing"] = export_config
    export_selection["mask_postprocessing_version"] = (
        "solar_streaming_candidate_guidance_nonoverlap_v4"
    )
    if export_config != SELECTED_CONFIG:
        # A previous score for overlapping instances does not describe this output.
        export_selection.pop("calibration_PQ", None)
        export_selection["recalibration_needed"] = True
        print(
            "Legacy instance mode replaced by disjoint union/components. Recalibrate for updated PQ.",
            flush=True,
        )

    with open(temporary, mode="w", newline="", encoding="utf-8") as f:
        # Exactly the original writer: default dialect, quoting and CRLF ending.
        writer = csv.writer(f)
        writer.writerow(["filament_id", "segmentation_rle"])

        def task(path):
            cached = cached_predict(engine, path, SELECTED_CACHE_DIR)
            rows = submission_rows_for_source(
                path.stem, final_prediction(cached, export_config)
            )
            if PREDICTION_CACHE_POLICY != "all" and path.stem not in preview_stems:
                delete_prediction_source(SELECTED_CACHE_DIR, path.stem)
            return path.stem, rows

        with InferenceEngine(SELECTED_CHECKPOINT) as engine:
            results = bounded_map(task, paths)
            for stem, worker_rows in tqdm(
                results, total=len(paths), desc="Submission, both GPUs"
            ):
                for fid, rle in worker_rows:
                    if fid in seen_filament_ids:
                        raise ValueError(f"Duplicate filament ID detected: {fid}")
                    seen_filament_ids.add(fid)
                    writer.writerow([fid, rle])
                processed[stem] = len(worker_rows)

    assert set(processed) == {path.stem for path in paths}
    audit = audit_submission(temporary, processed)
    os.replace(temporary, output_path)
    save_json(EXPERIMENT_DIR / "test_inference_log.json", processed)
    audit.update(
        sha256=file_sha256(output_path),
        inference_seconds=time.perf_counter() - start_time,
        csv=str(output_path),
        pipeline=export_selection,
        submission_format="original_0_36",
    )
    save_json(EXPERIMENT_DIR / "submission_audit.json", audit)
    print(
        f"\n[SUCCESS] Submission complete!\nProcessed: {len(paths):,} images"
        f"\nFilament rows written: {audit['submitted_filaments']:,}\nSaved: {output_path}",
        flush=True,
    )
    return audit


def create_submission(output_path=None):
    # Compatibility with the previous call name; serialization stays original.
    return create_submission_parallel(
        SUBMISSION_PATH if output_path is None else output_path
    )


def audit_submission(path, processed):
    seen = set()
    counts = {stem: 0 for stem in processed}
    # Compact accumulated RLE per image handles interleaved image rows as well.
    occupied_rles = {}
    with open(path, newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["filament_id", "segmentation_rle"]
        for row in reader:
            assert (
                set(row) == {"filament_id", "segmentation_rle"}
                and None not in row.values()
            )
            filament_id = row["filament_id"]
            stem, tail = filament_id.rsplit("_", 1)
            assert (
                stem in processed
                and tail.isdigit()
                and int(tail) > 0
                and filament_id not in seen
            )
            seen.add(filament_id)
            assert row["segmentation_rle"], "Empty RLE counts."
            rle = native_rle({"size": [2048, 2048], "counts": row["segmentation_rle"]})
            mask = decode_mask(rle)
            assert mask.shape == (2048, 2048) and mask.any()
            assert (
                mask_to_rle(mask) == row["segmentation_rle"]
            ), "Original-encoder RLE roundtrip failed."
            if stem in occupied_rles:
                intersection = mask_utils.merge(
                    [occupied_rles[stem], rle], intersect=True
                )
                overlap_pixels = int(mask_utils.area(intersection))
                if overlap_pixels:
                    raise ValueError(
                        f"Invalid submission: {filament_id} overlaps an earlier mask for {stem} "
                        f"by {overlap_pixels:,} pixels. The final CSV was not published."
                    )
                occupied_rles[stem] = mask_utils.merge([occupied_rles[stem], rle])
            else:
                occupied_rles[stem] = rle
            counts[stem] += 1
    assert counts == processed, "CSV rows differ from the complete test inference log."
    return dict(
        test_images_processed=len(processed),
        zero_prediction_images=sum(v == 0 for v in processed.values()),
        submitted_filaments=len(seen),
        decoded_every_rle=True,
        exact_columns=True,
        unique_ids=True,
        non_overlapping_masks=True,
        overlap_pixels=0,
    )


if CREATE_SUBMISSION:
    submission_audit = create_submission_parallel()
    print(json.dumps(submission_audit, indent=2))
else:
    print("CREATE_SUBMISSION=False: submission and test galleries skipped.")

## 10. Test previews of masks actually written to CSV

Select evenly spaced test filenames without consulting labels. Decode their final
masks from the audited CSV and display the same model input and cached predictions.
No second inference is needed. Preview maps remain temporary and are removed by the
completion cell below; PNG files are retained only if SAVE_PREVIEW_IMAGES=True.

In [ ]:
if CREATE_SUBMISSION:
    preview_paths = pick_evenly(test_image_paths(), PREDICTION_PREVIEW_COUNT)
    preview_rles = {p.stem: [] for p in preview_paths}
    with Path(SUBMISSION_PATH).open(newline="", encoding="utf-8") as stream:
        for row in csv.DictReader(stream):
            stem = row["filament_id"].rsplit("_", 1)[0]
            if stem in preview_rles:
                preview_rles[stem].append(
                    {"size": [2048, 2048], "counts": row["segmentation_rle"]}
                )
    for path in preview_paths:
        cached = cached_predict(None, path, SELECTED_CACHE_DIR)
        prediction_preview(path, cached, preview_rles[path.stem], "test_predictions")
    print(
        f"Displayed {len(preview_paths)} distinct test prediction previews."
    )

## 11. Keep results and remove completed intermediates

Run this cell after all report/test previews. It verifies the selected model and
frozen configuration, checks report results, and (when requested) verifies the
audited submission and its SHA-256 before removing generated data. If training,
inference, auditing, or previews fail, this completion cell is not reached in Run All.
The original Kaggle dataset is outside every cleanup target.

Retained: best/last weights (and requested epoch weights), training results.csv and
curves, calibration/report metrics, split CSVs, target/resource audits, run and
preprocessing configuration, embedded runtime source, selected_pipeline.json,
test inference/submission audits, and a_submission.csv. GPU utilization, batch plans, and input-throughput logs are also retained. Prepared
model-input arrays, prediction maps, and generated image aliases/labels are temporary. Source images and source annotations remain
untouched. Rerunning the notebook regenerates the small label workspace; retuning
after cleanup requires inference again. No public-score improvement is claimed.

In [ ]:
def cleanup_completed_run():
    if not CLEANUP_INTERMEDIATES:
        print('CLEANUP_INTERMEDIATES=False: generated intermediates retained.')
        return {'completed': False, 'reason': 'disabled'}
    selected_path = EXPERIMENT_DIR / 'selected_pipeline.json'
    report_path = EXPERIMENT_DIR / 'report_metrics.csv'
    if not selected_path.is_file() or not report_path.is_file():
        raise RuntimeError('Report/selection incomplete; temporary data was retained.')
    selected = json.loads(selected_path.read_text(encoding='utf-8'))
    weights = Path(selected['checkpoint'])
    if not weights.is_file() or file_sha256(weights) != selected['weights_sha256']:
        raise RuntimeError('Selected weights are missing/changed; temporary data was retained.')
    if selected['inference'] != current_inference_config():
        raise RuntimeError('Selected preprocessing changed; temporary data was retained.')
    if RUN_TRAINING and not (run_dir / 'results.csv').is_file():
        raise RuntimeError('Training history missing; temporary data was retained.')
    if CREATE_SUBMISSION:
        audit_path = EXPERIMENT_DIR / 'submission_audit.json'
        if not audit_path.is_file():
            raise RuntimeError('Submission audit missing; temporary data was retained.')
        audit = json.loads(audit_path.read_text(encoding='utf-8'))
        if (not Path(SUBMISSION_PATH).is_file() or not audit.get('non_overlapping_masks')
                or audit.get('sha256') != file_sha256(SUBMISSION_PATH)
                or audit.get('pipeline') != selected):
            raise RuntimeError('Submission verification failed; temporary data was retained.')
    # Validate every recursive target before making any deletion.
    dataset_root = checked_generated_directory(YOLO_DATA_DIR, 'yolo_filament_dataset')
    preview_root = checked_generated_directory(PREVIEW_DIR, 'previews')
    cache_root = checked_generated_directory(EXPERIMENT_DIR / 'prediction_cache', 'prediction_cache')
    for directory in list(cache_root.iterdir()) if cache_root.exists() else []:
        if directory.is_dir():
            checked_prediction_cache_dir(directory)
    removed_map_bytes = 0
    for directory in list(cache_root.iterdir()) if cache_root.exists() else []:
        if directory.is_dir():
            removed_map_bytes += prune_prediction_cache(directory)
            if not any(directory.iterdir()):
                directory.rmdir()
    if cache_root.exists() and not any(cache_root.iterdir()):
        cache_root.rmdir()
    removed_input_cache_bytes = directory_bytes(dataset_root / 'input_cache')
    remove_generated_dataset()
    if preview_root.exists() and not SAVE_PREVIEW_IMAGES:
        shutil.rmtree(preview_root)
    removed_batch_previews = []
    if RUN_TRAINING and run_dir.resolve().is_relative_to(RUNS_DIR.resolve()):
        for pattern in ('train_batch*.jpg', 'val_batch*.jpg', 'labels.jpg', 'labels_correlogram.jpg'):
            for path in run_dir.glob(pattern):
                if path.is_file() and not path.is_symlink():
                    path.unlink()
                    removed_batch_previews.append(path.name)
    result = dict(completed=True, removed_prediction_bytes=removed_map_bytes,
                  removed_input_cache_bytes=removed_input_cache_bytes,
                  removed_generated_dataset=str(dataset_root),
                  removed_training_batch_previews=removed_batch_previews,
                  preserved_weights=str(weights),
                  preserved_submission=str(SUBMISSION_PATH) if CREATE_SUBMISSION else None)
    save_json(EXPERIMENT_DIR / 'cleanup_report.json', result)
    gc.collect()
    if GPU_IDS:
        torch.cuda.empty_cache()
    print('Completed cleanup. Kept weights, history, metrics, configuration/runtime, and submission.')
    return result


cleanup_report = cleanup_completed_run()
# Release the source COCO Python object after every label-dependent stage is complete.
if cleanup_report.get('completed'):
    del coco
    gc.collect()